# NetraScan.AI — Multi-Transformer Cataract Classification and Explainable AI

**Research and implementation notebook · Curated GitHub edition**  
**Created by Mr. SHOBHIT JAIN**

## Project overview

NetraScan.AI is an academic research prototype for classifying eye images into **Normal**, **Immature Cataract**, and **Mature Cataract**. It evaluates three transformer-based architectures—**ViT-B/16**, **Swin-Tiny**, and **MobileViT-S**—and implements majority-vote inference, Swin-Tiny Grad-CAM visualization, and an offline structured report generator.

**Important:** This notebook preserves the original experimental logic, including the data-leakage investigation and the final V4 experiments. It is not a standalone, immediately executable GitHub notebook: original image datasets, V4 manifests, trained weights, and local paths are required. Code outputs have been cleared for repository hygiene.

**Research use only.** This software is not a validated medical device and must not be used to diagnose or triage patients.


## Study design and reported results

- **Data:** 3,542 images consolidated from two source collections, with a V4 family-aware train/validation/test partition of **2,454 / 517 / 571** images.
- **Leakage controls:** perceptual-hash similarity checks, augmentation-filename lineage inspection, and connected visual-family grouping; related images are assigned to the same partition.
- **Labels:** `normal`, `immature`, `mature`.
- **Selection and evaluation:** training and validation on V4, with a held-out 571-image test partition.

| Model | Held-out V4 test accuracy | Correct / total |
|---|---:|---:|
| ViT-B/16 | 97.37% | 556 / 571 |
| Swin-Tiny | 98.95% | 565 / 571 |
| MobileViT-S | 98.25% | 561 / 571 |

These are **internal held-out test** results, not independent external clinical validation. A high accuracy value does not establish generalization across patients, institutions, imaging equipment, or acquisition conditions. Model confidence is not a calibrated disease probability.

### Reproduction prerequisites

1. Python 3.11, compatible PyTorch/CUDA, torchvision, timm, pandas, scikit-learn, Pillow, matplotlib, Gradio, Plotly, and `grad-cam`.
2. Original image data and metadata under a local project directory; update the `ROOT` assignment in the notebook.
3. Pretrained ImageNet weights for training and the trained V4 checkpoints for inference.
4. A CUDA-capable GPU is recommended for training. Portable Windows runtime setup is documented separately from these research experiments.

**Execution note:** The historical leakage-audit section depends on intermediate data structures produced during the investigation. For a fresh reproducibility run, first restore the archived source manifests and datasets. The notebook is a curated research record, not a fully parameterized one-command training pipeline.


## 01 · Environment and initial dataset inspection

The original environment and first dataset manifests are retained for methodological provenance. The original notebook uses a Windows-specific `ROOT` path; update it before execution. CUDA-enabled PyTorch and `timm` must already be installed.


### Verify Python and PyTorch


In [ ]:
# STEP 01 — Verify Python and PyTorch
#cell 1
import sys
import importlib.util

print("Python version:", sys.version.split()[0])

try:
    import torch

    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA version:", torch.version.cuda)
        print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
    else:
        print("GPU acceleration unavailable")

except ImportError:
    print("PyTorch is not installed in this environment")

for package in ["timm", "torchvision", "pandas", "sklearn", "PIL", "matplotlib", "tqdm"]:
    status = "Installed" if importlib.util.find_spec(package) else "Missing"
    print(f"{package}: {status}")


### Confirm CUDA availability


In [ ]:
# STEP 02 — Confirm CUDA availability
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

### Inspect timm and torchvision


In [ ]:
# STEP 03 — Inspect timm and torchvision
#cell 4
import torch
import torchvision
import timm

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("timm:", timm.__version__)

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=3
).to("cuda")

print("ViT initialized successfully")
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

del model
torch.cuda.empty_cache()


### Load initial metadata


In [ ]:
# STEP 04 — Load initial metadata
#cell 5
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Personal AI & ML PROJECTS\AI IN HEALTHCARE\MY PROJECT E PAARVAI")
METADATA_DIR = ROOT / "NetraScan_Metadata"

train_df = pd.read_csv(METADATA_DIR / "netrascan_train.csv")
val_df = pd.read_csv(METADATA_DIR / "netrascan_val.csv")
test_df = pd.read_csv(METADATA_DIR / "netrascan_test.csv")

print("TRAIN:", train_df.shape)
print("VALIDATION:", val_df.shape)
print("TEST:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nClass distribution:")
for name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\n{name}")
    print(df["class"].value_counts())

print("\nChecking image paths:")
for name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    missing = sum(not Path(p).is_file() for p in df["image_path"])
    print(f"{name}: {missing} missing images")

print("\nChecking visual-family overlap:")
train_families = set(train_df["visual_family_id"])
val_families = set(val_df["visual_family_id"])
test_families = set(test_df["visual_family_id"])

print("Train/Val:", len(train_families & val_families))
print("Train/Test:", len(train_families & test_families))
print("Val/Test:", len(val_families & test_families))


### Define image dataset and preprocessing


In [ ]:
# STEP 05 — Define image dataset and preprocessing
#Dataset class and preprocessing
#cell 6


import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from timm.data import resolve_model_data_config

CLASS_NAMES = ["normal", "immature", "mature"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}

MODEL_NAME = "vit_base_patch16_224"

reference_model = timm.create_model(
    MODEL_NAME,
    pretrained=False,
    num_classes=3
)

data_config = resolve_model_data_config(reference_model)
del reference_model

image_size = data_config["input_size"][-1]
mean = data_config["mean"]
std = data_config["std"]

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomRotation(8),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(
        brightness=0.08,
        contrast=0.08,
        saturation=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

eval_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

class CataractDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image = Image.open(row["image_path"]).convert("RGB")
        image = self.transform(image)
        label = CLASS_TO_IDX[row["class"]]
        return image, label

train_dataset = CataractDataset(train_df, train_transform)
val_dataset = CataractDataset(val_df, eval_transform)
test_dataset = CataractDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Classes:", CLASS_TO_IDX)
print("Image size:", image_size)
print("Normalization mean:", mean)
print("Normalization std:", std)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


### Visualize a training batch


In [ ]:
# STEP 06 — Visualize a training batch
#cell 7 Display sample images

import matplotlib.pyplot as plt
import numpy as np

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label indices:", labels.tolist())

fig, axes = plt.subplots(2, 4, figsize=(13, 7))

mean_array = np.array(mean).reshape(3, 1, 1)
std_array = np.array(std).reshape(3, 1, 1)

for i, ax in enumerate(axes.flat):
    image = images[i].permute(1, 2, 0).numpy()
    image = image * std_array.reshape(1, 1, 3) + mean_array.reshape(1, 1, 3)
    image = np.clip(image, 0, 1)

    ax.imshow(image)
    ax.set_title(IDX_TO_CLASS[labels[i].item()])
    ax.axis("off")

plt.suptitle("NetraScan.AI — Training Batch Inspection", fontsize=15)
plt.tight_layout()
plt.show()


### Initialize ViT-B/16


In [ ]:
# STEP 07 — Initialize ViT-B/16
#CELL 8
import torch
import timm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

MODEL_NAME = "vit_base_patch16_224"

model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=3
)

model = model.to(DEVICE)

print("Device:", DEVICE)
print("Model:", MODEL_NAME)
print("Output classes:", model.num_classes)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Verify the forward pass using a small batch
model.eval()

with torch.inference_mode():
    sample_batch = images[:2].to(DEVICE)
    outputs = model(sample_batch)

print("Input shape:", sample_batch.shape)
print("Output shape:", outputs.shape)

assert outputs.shape == (2, 3)

print("Pretrained ViT-B/16 loaded successfully!")


### Configure training


In [ ]:
# STEP 08 — Configure training
#CELL 9 Training configuration

import random
import numpy as np
import torch.nn as nn
from torch.optim import AdamW
from torch.amp import GradScaler

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 3

criterion = nn.CrossEntropyLoss()

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scaler = GradScaler("cuda")

CHECKPOINT_DIR = ROOT / "models" / "vit"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = ROOT / "results" / "vit"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Training configuration ready")
print("Device:", DEVICE)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", train_loader.batch_size)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Results directory:", RESULTS_DIR)


### Benchmark training memory


In [ ]:
# STEP 09 — Benchmark training memory
#cell 10 single step training gpu benchmark test 

import time
from torch.amp import autocast

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model.train()

batch_images, batch_labels = next(iter(train_loader))
batch_images = batch_images.to(DEVICE, non_blocking=True)
batch_labels = batch_labels.to(DEVICE, non_blocking=True)

print("Batch shape:", batch_images.shape)
print("Starting GPU memory benchmark...")

try:
    torch.cuda.synchronize()
    start_time = time.perf_counter()

    optimizer.zero_grad(set_to_none=True)

    with autocast(device_type="cuda", dtype=torch.float16):
        outputs = model(batch_images)
        loss = criterion(outputs, batch_labels)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time

    peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)
    reserved_memory = torch.cuda.max_memory_reserved() / (1024 ** 3)

    print("\nBENCHMARK SUCCESSFUL")
    print(f"Loss: {loss.item():.4f}")
    print(f"Step time: {elapsed:.2f} seconds")
    print(f"Peak allocated VRAM: {peak_memory:.2f} GB")
    print(f"Peak reserved VRAM: {reserved_memory:.2f} GB")
    print(f"GPU total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024 ** 3):.2f} GB")

except torch.cuda.OutOfMemoryError:
    print("\nCUDA OUT OF MEMORY")
    print("Batch size 16 is too large for this configuration.")
    print("We will reduce the batch size to 8.")

finally:
    optimizer.zero_grad(set_to_none=True)

    del batch_images, batch_labels

    if "outputs" in globals():
        del outputs

    torch.cuda.empty_cache()



### Define training and validation loops


In [ ]:
# STEP 10 — Define training and validation loops
#cell 11 Training and validation functions

import time
import torch
import numpy as np

from torch.amp import autocast
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm


def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    progress = tqdm(loader, desc="Training", leave=False)

    for batch_images, batch_labels in progress:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(batch_images)
            loss = criterion(outputs, batch_labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * batch_images.size(0)

        predictions = outputs.detach().argmax(dim=1)

        all_preds.extend(predictions.cpu().tolist())
        all_labels.extend(batch_labels.cpu().tolist())

        progress.set_postfix(loss=f"{loss.item():.4f}")

    epoch_loss = total_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(
        all_labels, all_preds, average="macro", zero_division=0
    )

    return epoch_loss, epoch_acc, epoch_f1


@torch.inference_mode()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    progress = tqdm(loader, desc="Validation", leave=False)

    for batch_images, batch_labels in progress:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)

        with autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(batch_images)
            loss = criterion(outputs, batch_labels)

        total_loss += loss.item() * batch_images.size(0)

        predictions = outputs.argmax(dim=1)

        all_preds.extend(predictions.cpu().tolist())
        all_labels.extend(batch_labels.cpu().tolist())

    epoch_loss = total_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(
        all_labels, all_preds, average="macro", zero_division=0
    )

    return epoch_loss, epoch_acc, epoch_f1


print("Training and validation functions initialized.")


## 02 · Initial ViT experiment and leakage investigation

The initial validation results prompted an investigation into near-duplicate images and augmentation lineage. **Do not treat the initial validation accuracy as a leakage-free performance estimate.** These cells document the audit and subsequent corrective splitting procedure.


### Initial ViT experiment


In [ ]:
# STEP 11 — Initial ViT experiment
#cell 12 Full ViT training

import json
import random
import time
import pandas as pd
import numpy as np
import torch
import timm

from torch.optim import AdamW
from torch.amp import GradScaler

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Reset the model after our benchmark
del model
torch.cuda.empty_cache()

model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=3
).to(DEVICE)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scaler = GradScaler("cuda")

# Save experiment configuration
config = {
    "model": MODEL_NAME,
    "pretrained": True,
    "classes": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,
    "image_size": image_size,
    "normalization_mean": list(mean),
    "normalization_std": list(std),
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "test_samples": len(test_dataset),
    "batch_size": train_loader.batch_size,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "max_epochs": EPOCHS,
    "early_stopping_patience": PATIENCE,
    "seed": SEED,
    "optimizer": "AdamW",
    "loss": "CrossEntropyLoss",
    "mixed_precision": "FP16",
    "selection_metric": "validation_macro_f1"
}

with open(RESULTS_DIR / "experiment_config.json", "w") as f:
    json.dump(config, f, indent=4)

best_f1 = -1.0
patience_counter = 0
history = []

print("Starting ViT-B/16 fine-tuning")
print("Device:", DEVICE)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("-" * 65)

for epoch in range(EPOCHS):
    epoch_start = time.perf_counter()

    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, DEVICE
    )

    val_loss, val_acc, val_f1 = validate_one_epoch(
        model, val_loader, criterion, DEVICE
    )

    elapsed = time.perf_counter() - epoch_start

    metrics = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_macro_f1": train_f1,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1,
        "epoch_seconds": elapsed
    }

    history.append(metrics)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Train F1: {train_f1:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    # Save history after every epoch
    pd.DataFrame(history).to_csv(
        RESULTS_DIR / "training_history.csv",
        index=False
    )

    if val_f1 > best_f1 + 1e-4:
        best_f1 = val_f1
        patience_counter = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_macro_f1": val_f1,
                "class_to_idx": CLASS_TO_IDX,
                "model_name": MODEL_NAME
            },
            CHECKPOINT_DIR / "vit_best.pth"
        )

        print(f"  Best checkpoint saved — Val Macro-F1: {best_f1:.4f}")

    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{PATIENCE})")

    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

print("-" * 65)
print("Training completed.")
print(f"Best validation macro-F1: {best_f1:.4f}")
print("Checkpoint:", CHECKPOINT_DIR / "vit_best.pth")
print("History:", RESULTS_DIR / "training_history.csv")


### Audit cross-split pHash similarity


In [ ]:
# STEP 12 — Audit cross-split pHash similarity
# cell 13 Audit cross-split near-duplicates

from collections import Counter
import numpy as np
import pandas as pd

all_splits = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

hashes = np.array(
    [int(h, 16) for h in all_splits["phash"]],
    dtype=np.uint64
)

split_labels = all_splits["split"].to_numpy()

cross_split_pairs = []

for i in range(len(hashes)):
    for j in range(i + 1, len(hashes)):
        if split_labels[i] == split_labels[j]:
            continue

        distance = (
            int(hashes[i]) ^ int(hashes[j])
        ).bit_count()

        if distance <= 4:
            cross_split_pairs.append({
                "index_1": i,
                "index_2": j,
                "split_1": split_labels[i],
                "split_2": split_labels[j],
                "class_1": all_splits.iloc[i]["class"],
                "class_2": all_splits.iloc[j]["class"],
                "distance": distance
            })

audit_df = pd.DataFrame(cross_split_pairs)

print("Cross-split near-duplicate candidates:", len(audit_df))

if not audit_df.empty:
    print("\nDistance distribution:")
    print(audit_df["distance"].value_counts().sort_index())

    print("\nClass agreement:")
    print(
        (audit_df["class_1"] == audit_df["class_2"])
        .value_counts()
    )

    display(audit_df.head(10))


### Inspect near-duplicate images


In [ ]:
# STEP 13 — Inspect near-duplicate images
#cell 14 Cross-split visual inspection

import matplotlib.pyplot as plt
from PIL import Image

# Prioritize the closest perceptual-hash matches
inspection_df = (
    audit_df
    .sort_values("distance")
    .drop_duplicates(subset=["index_1"])
    .head(12)
)

fig, axes = plt.subplots(
    len(inspection_df), 2,
    figsize=(9, len(inspection_df) * 3)
)

for row_num, (_, pair) in enumerate(inspection_df.iterrows()):
    idx1 = int(pair["index_1"])
    idx2 = int(pair["index_2"])

    record1 = all_splits.iloc[idx1]
    record2 = all_splits.iloc[idx2]

    for col, record in enumerate([record1, record2]):
        image = Image.open(record["image_path"]).convert("RGB")

        axes[row_num, col].imshow(image)
        axes[row_num, col].axis("off")

        axes[row_num, col].set_title(
            f"{record['split']} | {record['class']}\n"
            f"{record['source']} | {record['filename'][:30]}",
            fontsize=8
        )

    axes[row_num, 0].set_ylabel(
        f"pHash distance: {int(pair['distance'])}",
        fontsize=9
    )

plt.suptitle(
    "NetraScan.AI — Cross-Split Near-Duplicate Inspection",
    fontsize=14
)

plt.tight_layout()
plt.show()


### Audit augmentation filename lineage


In [ ]:
# STEP 14 — Audit augmentation filename lineage
#cell 15 Filename lineage audit

import re
import pandas as pd

lineage_df = all_splits.copy()

def extract_original_stem(row):
    filename = str(row["filename"])
    source = str(row["source"])

    stem = filename.rsplit(".", 1)[0]

    if source == "cataract_v01":
        stem = re.sub(
            r"\.rf\.[a-fA-F0-9]+$",
            "",
            stem
        )

    return stem.lower()

lineage_df["original_stem"] = lineage_df.apply(
    extract_original_stem,
    axis=1
)

# Include class and source to avoid unrelated filename collisions
lineage_df["lineage_key"] = (
    lineage_df["source"].astype(str)
    + "::"
    + lineage_df["class"].astype(str)
    + "::"
    + lineage_df["original_stem"]
)

family_summary = (
    lineage_df.groupby("lineage_key")
    .agg(
        images=("filename", "size"),
        splits=("split", "nunique"),
        sources=("source", "nunique")
    )
    .reset_index()
)

cross_split_lineages = family_summary[
    family_summary["splits"] > 1
]

print("Total filename lineage groups:", len(family_summary))
print("Cross-split filename lineage groups:", len(cross_split_lineages))

print("\nImages in cross-split lineages:")
print(
    lineage_df[
        lineage_df["lineage_key"].isin(
            cross_split_lineages["lineage_key"]
        )
    ].shape[0]
)

display(
    cross_split_lineages
    .sort_values("images", ascending=False)
    .head(20)
)


### Construct provisional image families


In [ ]:
# STEP 15 — Construct provisional image families
#cell 16 Union-Find grouping with filename lineage

from collections import defaultdict, Counter
import numpy as np
import pandas as pd

# Start from our existing combined dataset
corrected_df = lineage_df.copy().reset_index(drop=True)

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        root_a = self.find(a)
        root_b = self.find(b)

        if root_a == root_b:
            return

        if self.rank[root_a] < self.rank[root_b]:
            root_a, root_b = root_b, root_a

        self.parent[root_b] = root_a

        if self.rank[root_a] == self.rank[root_b]:
            self.rank[root_a] += 1


uf = UnionFind(len(corrected_df))

# 1. Preserve original pHash=0 visual families
for _, group in corrected_df.groupby("visual_family_id"):
    indices = group.index.tolist()
    for idx in indices[1:]:
        uf.union(indices[0], idx)

# 2. Merge Roboflow augmentation variants sharing original filename
roboflow_groups = corrected_df[
    corrected_df["source"] == "cataract_v01"
].groupby("lineage_key")

for _, group in roboflow_groups:
    indices = group.index.tolist()
    for idx in indices[1:]:
        uf.union(indices[0], idx)

# 3. Merge pHash distance <= 2 within matching classes
hashes = [
    int(h, 16) for h in corrected_df["phash"]
]
labels = corrected_df["class"].tolist()

merged_pairs = 0

for i in range(len(corrected_df)):
    for j in range(i + 1, len(corrected_df)):
        if labels[i] != labels[j]:
            continue

        distance = (hashes[i] ^ hashes[j]).bit_count()

        if distance <= 2:
            uf.union(i, j)
            merged_pairs += 1

# Assign new visual-family IDs
roots = [uf.find(i) for i in range(len(corrected_df))]
unique_roots = sorted(set(roots))

root_to_id = {
    root: f"VF2_{idx + 1:05d}"
    for idx, root in enumerate(unique_roots)
}

corrected_df["visual_family_id_v2"] = [
    root_to_id[root] for root in roots
]

family_sizes = corrected_df[
    "visual_family_id_v2"
].value_counts()

corrected_df["visual_family_size_v2"] = (
    corrected_df["visual_family_id_v2"].map(family_sizes)
)

print("Total images:", len(corrected_df))
print("Original visual families:", corrected_df["visual_family_id"].nunique())
print("Corrected visual families:", corrected_df["visual_family_id_v2"].nunique())
print("pHash <= 2 candidate pairs:", merged_pairs)
print("Largest corrected family:", family_sizes.max())

conflicts = (
    corrected_df.groupby("visual_family_id_v2")["class"]
    .nunique()
)

print("Families with conflicting labels:", (conflicts > 1).sum())

# Check how many corrected families span original splits
cross_split_families = (
    corrected_df.groupby("visual_family_id_v2")["split"]
    .nunique()
)

print(
    "Corrected families spanning original splits:",
    (cross_split_families > 1).sum()
)


### Generate corrected V2 split


In [ ]:
# STEP 16 — Generate corrected V2 split
#cell 17 Corrected group-aware splitting


from sklearn.model_selection import train_test_split
import pandas as pd

# One representative record per corrected visual family
family_df = (
    corrected_df.groupby("visual_family_id_v2")
    .agg(
        class_label=("class", "first"),
        family_size=("visual_family_size_v2", "first")
    )
    .reset_index()
)

# 70% train, 30% temporary
train_families, temp_families = train_test_split(
    family_df,
    test_size=0.30,
    stratify=family_df["class_label"],
    random_state=42
)

# Divide remaining 30% into 15% validation and 15% test
val_families, test_families = train_test_split(
    temp_families,
    test_size=0.50,
    stratify=temp_families["class_label"],
    random_state=42
)

# Map family IDs to their assigned splits
split_mapping = {}

for family_id in train_families["visual_family_id_v2"]:
    split_mapping[family_id] = "train"

for family_id in val_families["visual_family_id_v2"]:
    split_mapping[family_id] = "val"

for family_id in test_families["visual_family_id_v2"]:
    split_mapping[family_id] = "test"

corrected_df["split_v2"] = (
    corrected_df["visual_family_id_v2"].map(split_mapping)
)

print("Corrected split distribution:")
print(corrected_df["split_v2"].value_counts())

print("\nClass distribution:")
print(
    pd.crosstab(
        corrected_df["split_v2"],
        corrected_df["class"]
    )
)

# Verify corrected visual families never cross splits
leakage_check = (
    corrected_df.groupby("visual_family_id_v2")["split_v2"]
    .nunique()
)

print("\nCorrected families crossing splits:",
      (leakage_check > 1).sum())

assert (leakage_check > 1).sum() == 0

print("\nPASS — No corrected visual-family overlap across splits.")


### Audit V2 similarity


In [ ]:
# STEP 17 — Audit V2 similarity
#cell 18 V2 split-integrity audit


from collections import Counter
import pandas as pd

# Work only with the corrected in-memory dataset
v2_audit = corrected_df.reset_index(drop=True).copy()

hashes_v2 = [
    int(h, 16) for h in v2_audit["phash"]
]

remaining_pairs = []

for i in range(len(v2_audit)):
    for j in range(i + 1, len(v2_audit)):
        if v2_audit.at[i, "split_v2"] == v2_audit.at[j, "split_v2"]:
            continue

        distance = (
            hashes_v2[i] ^ hashes_v2[j]
        ).bit_count()

        if distance <= 4:
            remaining_pairs.append({
                "index_1": i,
                "index_2": j,
                "distance": distance,
                "split_1": v2_audit.at[i, "split_v2"],
                "split_2": v2_audit.at[j, "split_v2"],
                "class_1": v2_audit.at[i, "class"],
                "class_2": v2_audit.at[j, "class"],
                "source_1": v2_audit.at[i, "source"],
                "source_2": v2_audit.at[j, "source"]
            })

remaining_df = pd.DataFrame(remaining_pairs)

print("Remaining cross-split pHash candidates:", len(remaining_df))

if not remaining_df.empty:
    print("\nDistance distribution:")
    print(remaining_df["distance"].value_counts().sort_index())

    print("\nClass agreement:")
    print(
        (remaining_df["class_1"] == remaining_df["class_2"])
        .value_counts()
    )

    print("\nSource combinations:")
    print(
        pd.crosstab(
            remaining_df["source_1"],
            remaining_df["source_2"]
        )
    )

    display(remaining_df.head(15))

# Check Roboflow augmentation lineage
roboflow_v2 = v2_audit[
    v2_audit["source"] == "cataract_v01"
]

lineage_overlap = (
    roboflow_v2.groupby("lineage_key")["split_v2"]
    .nunique()
)

print(
    "\nRoboflow lineages crossing V2 splits:",
    int((lineage_overlap > 1).sum())
)

print(
    "V2 families crossing splits:",
    int(
        (
            v2_audit.groupby("visual_family_id_v2")["split_v2"]
            .nunique() > 1
        ).sum()
    )
)


### Inspect remaining candidates


In [ ]:
# STEP 18 — Inspect remaining candidates
import matplotlib.pyplot as plt
from PIL import Image

# Select representative pairs from each dataset
inspection = []

for source in ["cataract_v01", "dataset_katarak"]:
    subset = remaining_df[
        (remaining_df["source_1"] == source) &
        (remaining_df["source_2"] == source)
    ].copy()

    subset["family_1"] = subset["index_1"].map(
        v2_audit["visual_family_id_v2"]
    )
    subset["family_2"] = subset["index_2"].map(
        v2_audit["visual_family_id_v2"]
    )

    subset["family_pair"] = subset.apply(
        lambda row: tuple(sorted([
            row["family_1"],
            row["family_2"]
        ])),
        axis=1
    )

    subset = subset.drop_duplicates("family_pair")
    inspection.append(subset.head(8))

inspection_df = pd.concat(
    inspection, ignore_index=True
)

print("Representative family pairs:", len(inspection_df))

if len(inspection_df) > 0:
    fig, axes = plt.subplots(
        len(inspection_df), 2,
        figsize=(10, len(inspection_df) * 2.8),
        squeeze=False
    )

    for row_num, (_, pair) in enumerate(inspection_df.iterrows()):
        idx1 = int(pair["index_1"])
        idx2 = int(pair["index_2"])

        for col, idx in enumerate([idx1, idx2]):
            record = v2_audit.iloc[idx]

            with Image.open(record["image_path"]) as img:
                image = img.convert("RGB")

            axes[row_num, col].imshow(image)
            axes[row_num, col].axis("off")
            axes[row_num, col].set_title(
                f"{record['source']} | {record['split_v2']}\n"
                f"{record['class']} | {record['filename'][:35]}",
                fontsize=8
            )

        axes[row_num, 0].set_ylabel(
            f"Pair {row_num + 1}\npHash = {pair['distance']}",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()


### Merge V3 image families


In [ ]:
# STEP 19 — Merge V3 image families
# Build V3 groups from the existing V2 groups
uf_v3 = UnionFind(len(v2_audit))

# Preserve every V2 visual family
for _, group in v2_audit.groupby("visual_family_id_v2"):
    indices = group.index.tolist()

    for idx in indices[1:]:
        uf_v3.union(indices[0], idx)

# Merge all same-class pHash distance-4 candidates
merged_candidates = 0

for _, pair in remaining_df.iterrows():
    i = int(pair["index_1"])
    j = int(pair["index_2"])

    if (
        pair["distance"] == 4
        and pair["class_1"] == pair["class_2"]
    ):
        uf_v3.union(i, j)
        merged_candidates += 1

roots_v3 = [
    uf_v3.find(i)
    for i in range(len(v2_audit))
]

unique_roots_v3 = sorted(set(roots_v3))

root_to_v3 = {
    root: f"VF3_{n + 1:05d}"
    for n, root in enumerate(unique_roots_v3)
}

v3_df = v2_audit.copy()

v3_df["visual_family_id_v3"] = [
    root_to_v3[root]
    for root in roots_v3
]

v3_sizes = v3_df["visual_family_id_v3"].value_counts()

v3_df["visual_family_size_v3"] = (
    v3_df["visual_family_id_v3"].map(v3_sizes)
)

print("Total images:", len(v3_df))
print("V2 families:", v2_audit["visual_family_id_v2"].nunique())
print("V3 families:", v3_df["visual_family_id_v3"].nunique())
print("Candidate pairs grouped:", merged_candidates)
print("Largest V3 family:", v3_sizes.max())

conflicts = (
    v3_df.groupby("visual_family_id_v3")["class"]
    .nunique()
)

print("Conflicting-label families:", (conflicts > 1).sum())

assert (conflicts > 1).sum() == 0

print("\nPASS — V3 grouping constructed successfully.")


### Create V3 split


In [ ]:
# STEP 20 — Create V3 split
#Cell 20 — V3 stratified splitting
from sklearn.model_selection import train_test_split
import pandas as pd

RANDOM_SEED = 42

# One record per V3 visual family
v3_family_df = (
    v3_df.groupby("visual_family_id_v3")
    .agg(
        class_label=("class", "first"),
        family_size=("visual_family_size_v3", "first")
    )
    .reset_index()
)

# Split families: 70% train, 30% temporary
v3_train_families, v3_temp_families = train_test_split(
    v3_family_df,
    test_size=0.30,
    stratify=v3_family_df["class_label"],
    random_state=RANDOM_SEED
)

# Split temporary families equally into validation and test
v3_val_families, v3_test_families = train_test_split(
    v3_temp_families,
    test_size=0.50,
    stratify=v3_temp_families["class_label"],
    random_state=RANDOM_SEED
)

# Build family-to-split mapping
v3_split_map = {}

for family_id in v3_train_families["visual_family_id_v3"]:
    v3_split_map[family_id] = "train"

for family_id in v3_val_families["visual_family_id_v3"]:
    v3_split_map[family_id] = "val"

for family_id in v3_test_families["visual_family_id_v3"]:
    v3_split_map[family_id] = "test"

# Assign splits to all images
v3_df["split_v3"] = (
    v3_df["visual_family_id_v3"].map(v3_split_map)
)

# Display distributions
print("V3 dataset distribution:")
print(v3_df["split_v3"].value_counts())

print("\nV3 class distribution:")
print(
    pd.crosstab(
        v3_df["split_v3"],
        v3_df["class"]
    )
)

# Verify group isolation
v3_overlap = (
    v3_df.groupby("visual_family_id_v3")["split_v3"]
    .nunique()
)

print(
    "\nV3 families crossing splits:",
    int((v3_overlap > 1).sum())
)

assert (v3_overlap > 1).sum() == 0

print("\nPASS — V3 visual-family split isolation verified.")


### Audit V3 split


In [ ]:
# STEP 21 — Audit V3 split
#Cell 21 — Final V3 integrity audit

from pathlib import Path
from collections import Counter
import pandas as pd

audit = v3_df.reset_index(drop=True).copy()

hash_values = [int(h, 16) for h in audit["phash"]]
labels = audit["class"].tolist()
splits = audit["split_v3"].tolist()

cross_split_pairs = []
all_candidate_pairs = []

for i in range(len(audit)):
    for j in range(i + 1, len(audit)):

        if labels[i] != labels[j]:
            continue

        distance = (hash_values[i] ^ hash_values[j]).bit_count()

        if distance <= 4:
            pair = {
                "index_1": i,
                "index_2": j,
                "distance": distance,
                "split_1": splits[i],
                "split_2": splits[j]
            }

            all_candidate_pairs.append(pair)

            if splits[i] != splits[j]:
                cross_split_pairs.append(pair)

all_candidates_df = pd.DataFrame(all_candidate_pairs)
cross_split_v3_df = pd.DataFrame(cross_split_pairs)

print("Total same-class pHash <=4 pairs:", len(all_candidates_df))
print("Cross-split pHash <=4 pairs:", len(cross_split_v3_df))

if not cross_split_v3_df.empty:
    print("\nCross-split distance distribution:")
    print(
        cross_split_v3_df["distance"]
        .value_counts()
        .sort_index()
    )

# Roboflow filename lineage
roboflow = audit[audit["source"] == "cataract_v01"]

lineage_overlap = (
    roboflow.groupby("lineage_key")["split_v3"].nunique()
)

print(
    "\nRoboflow lineages crossing splits:",
    int((lineage_overlap > 1).sum())
)

# V3 family overlap
family_overlap = (
    audit.groupby("visual_family_id_v3")["split_v3"].nunique()
)

print(
    "V3 families crossing splits:",
    int((family_overlap > 1).sum())
)

# Missing images
missing_paths = sum(
    not Path(path).is_file()
    for path in audit["image_path"]
)

print("Missing image paths:", missing_paths)

print("\nAudit complete.")


### Build global V4 image families


In [ ]:
# STEP 22 — Build global V4 image families
#Cell 22 — Global Union-Find grouping

# GLOBAL VISUAL-FAMILY GROUPING
# No files will be modified or overwritten.

global_df = v3_df.reset_index(drop=True).copy()

uf_global = UnionFind(len(global_df))

# Preserve existing V3 families
for _, group in global_df.groupby("visual_family_id_v3"):
    indices = group.index.tolist()

    for idx in indices[1:]:
        uf_global.union(indices[0], idx)

# Merge all same-class pHash <=4 candidate relationships
global_pairs = 0

for _, pair in all_candidates_df.iterrows():
    i = int(pair["index_1"])
    j = int(pair["index_2"])

    # Verify indices and labels
    assert global_df.at[i, "class"] == global_df.at[j, "class"]

    uf_global.union(i, j)
    global_pairs += 1

# Assign global family IDs
roots = [uf_global.find(i) for i in range(len(global_df))]

root_to_global_id = {
    root: f"VFG_{n + 1:05d}"
    for n, root in enumerate(sorted(set(roots)))
}

global_df["visual_family_id_v4"] = [
    root_to_global_id[root]
    for root in roots
]

family_sizes = global_df["visual_family_id_v4"].value_counts()

global_df["visual_family_size_v4"] = (
    global_df["visual_family_id_v4"].map(family_sizes)
)

# Validate class consistency
class_conflicts = (
    global_df.groupby("visual_family_id_v4")["class"]
    .nunique()
)

print("Total images:", len(global_df))
print("V3 families:", global_df["visual_family_id_v3"].nunique())
print("V4 families:", global_df["visual_family_id_v4"].nunique())
print("Candidate pairs processed:", global_pairs)
print("Largest V4 family:", family_sizes.max())
print("Families with conflicting labels:", int((class_conflicts > 1).sum()))

print("\nV4 family size distribution:")
print(family_sizes.value_counts().sort_index().tail(15))

assert (class_conflicts > 1).sum() == 0

print("\nPASS — Global candidate-family grouping complete.")


### Create family-aware V4 split


In [ ]:
# STEP 23 — Create family-aware V4 split
#Cell 23 — V4 stratified splitting and verification

from sklearn.model_selection import train_test_split
from pathlib import Path
import pandas as pd

RANDOM_SEED = 42

# One representative row per V4 family
v4_family_df = (
    global_df.groupby("visual_family_id_v4")
    .agg(
        class_label=("class", "first"),
        family_size=("visual_family_size_v4", "first")
    )
    .reset_index()
)

# 70% train / 30% temporary
train_groups, temp_groups = train_test_split(
    v4_family_df,
    test_size=0.30,
    stratify=v4_family_df["class_label"],
    random_state=RANDOM_SEED
)

# 15% validation / 15% test
val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.50,
    stratify=temp_groups["class_label"],
    random_state=RANDOM_SEED
)

split_map = {}

for group_id in train_groups["visual_family_id_v4"]:
    split_map[group_id] = "train"

for group_id in val_groups["visual_family_id_v4"]:
    split_map[group_id] = "val"

for group_id in test_groups["visual_family_id_v4"]:
    split_map[group_id] = "test"

v4_df = global_df.copy()

v4_df["split_v4"] = (
    v4_df["visual_family_id_v4"].map(split_map)
)

print("V4 split distribution:")
print(v4_df["split_v4"].value_counts())

print("\nV4 class distribution:")
print(pd.crosstab(v4_df["split_v4"], v4_df["class"]))

# 1. Visual-family overlap
family_overlap = (
    v4_df.groupby("visual_family_id_v4")["split_v4"]
    .nunique()
)

# 2. Cross-split pHash <=4 candidates
# all_candidates_df indices correspond to global_df/v4_df
cross_split_candidates = sum(
    v4_df.iloc[int(row["index_1"])]["split_v4"]
    != v4_df.iloc[int(row["index_2"])]["split_v4"]
    for _, row in all_candidates_df.iterrows()
)

# 3. Roboflow augmentation lineage
roboflow_v4 = v4_df[
    v4_df["source"] == "cataract_v01"
]

lineage_overlap = (
    roboflow_v4.groupby("lineage_key")["split_v4"]
    .nunique()
)

# 4. Missing image paths
missing_images = sum(
    not Path(path).is_file()
    for path in v4_df["image_path"]
)

print("\nINTEGRITY CHECKS")
print("-" * 45)

print("Families crossing splits:",
      int((family_overlap > 1).sum()))

print("pHash <=4 pairs crossing splits:",
      cross_split_candidates)

print("Roboflow lineages crossing splits:",
      int((lineage_overlap > 1).sum()))

print("Missing image paths:", missing_images)

assert (family_overlap > 1).sum() == 0
assert cross_split_candidates == 0
assert (lineage_overlap > 1).sum() == 0
assert missing_images == 0

print("\nPASS — V4 split integrity checks successful.")


### Export V4 manifests


In [ ]:
# STEP 24 — Export V4 manifests
from pathlib import Path
import pandas as pd
import json

V4_DIR = ROOT / "NetraScan_Metadata" / "v4"
V4_DIR.mkdir(parents=True, exist_ok=True)

# Preserve the original metadata, plus V4 grouping and split assignments.
export_columns = [
    "source",
    "original_split",
    "filename",
    "class",
    "image_path",
    "sha256",
    "phash",
    "visual_family_id_v4",
    "visual_family_size_v4",
    "split_v4"
]

master_v4 = v4_df[export_columns].copy()

# Standardize column names for reuse in the training pipeline.
master_v4 = master_v4.rename(columns={
    "visual_family_id_v4": "visual_family_id",
    "visual_family_size_v4": "visual_family_size",
    "split_v4": "split"
})

master_v4.to_csv(
    V4_DIR / "netrascan_3class_master_v4.csv",
    index=False
)

for split_name in ["train", "val", "test"]:
    subset = master_v4[
        master_v4["split"] == split_name
    ].copy()

    subset.to_csv(
        V4_DIR / f"netrascan_{split_name}_v4.csv",
        index=False
    )

# Record the methodology for reproducibility.
split_config = {
    "version": "v4",
    "seed": 42,
    "target_split_ratio": {
        "train": 0.70,
        "val": 0.15,
        "test": 0.15
    },
    "grouping": [
        "Original visual-family relationships",
        "Roboflow filename augmentation lineage",
        "Global same-class pHash Hamming distance <= 4",
        "Union-Find connected components"
    ],
    "grouping_note": (
        "Similarity-based groups for split isolation; "
        "not verified patient identities."
    ),
    "total_images": len(master_v4),
    "visual_families": master_v4["visual_family_id"].nunique(),
    "split_counts": {
        k: int(v)
        for k, v in master_v4["split"].value_counts().items()
    }
}

with open(
    V4_DIR / "split_config_v4.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(split_config, f, indent=4)

print("V4 manifests saved successfully!\n")

for file in sorted(V4_DIR.iterdir()):
    print(file.name)

print("\nTotal images:", len(master_v4))
print("Visual families:", master_v4["visual_family_id"].nunique())
print("Split counts:", master_v4["split"].value_counts().to_dict())


## 03 · Corrected V4 data preparation

The final V4 manifests are the basis for the comparative experiments. Image-family grouping is designed to keep related photographs within a single partition. The split is family-aware, not an independent-patient or multi-hospital external validation.


### Load V4 manifests


In [ ]:
# STEP 25 — Load V4 manifests
# STEP 27 — Load corrected V4 dataset

import pandas as pd

V4_DIR = ROOT / "NetraScan_Metadata" / "v4"

train_df = pd.read_csv(V4_DIR / "netrascan_train_v4.csv")
val_df = pd.read_csv(V4_DIR / "netrascan_val_v4.csv")
test_df = pd.read_csv(V4_DIR / "netrascan_test_v4.csv")

print("V4 TRAIN:", train_df.shape)
print("V4 VALIDATION:", val_df.shape)
print("V4 TEST:", test_df.shape)

print("\nClass distribution:")
for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}")
    print(df["class"].value_counts())

print("\nMissing image paths:")
for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    missing = (~df["image_path"].map(
        lambda p: Path(p).is_file()
    )).sum()
    print(f"{name}: {missing}")


### Define V4 transformations


In [ ]:
# STEP 26 — Define V4 transformations
#Define the image transformations

from torchvision import transforms

# Training: mild augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(degrees=8),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

# Validation and testing: deterministic preprocessing
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

test_transform = val_transform

print("Training transforms: READY")
print("Validation transforms: READY")
print("Testing transforms: READY")


### Construct V4 data loaders


In [ ]:
# STEP 27 — Construct V4 data loaders
# STEP 28 — Build V4 datasets and DataLoaders

from torch.utils.data import DataLoader

BATCH_SIZE = 16
NUM_WORKERS = 0

# Reuse existing CataractDataset class and transforms
train_dataset = CataractDataset(
    train_df,
    transform=train_transform
)

val_dataset = CataractDataset(
    val_df,
    transform=val_transform
)

test_dataset = CataractDataset(
    test_df,
    transform=test_transform
)

# Recreate DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("V4 DATASET CONFIGURATION")
print("-" * 40)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

print("\nTraining batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

# Verify one training batch
images, labels = next(iter(train_loader))

print("\nBatch image shape:", images.shape)
print("Batch label shape:", labels.shape)

assert images.shape == (16, 3, 224, 224)
assert labels.shape == (16,)

print("\nPASS — V4 DataLoaders ready.")


## 04 · ViT-B/16: fine-tuning and evaluation

Train the ViT-B/16 model on V4, evaluate on validation and the held-out test set, inspect errors, and save figures. Reported test accuracy: **97.37% (556/571)**.


### Initialize V4 ViT-B/16


In [ ]:
# STEP 28 — Initialize V4 ViT-B/16
#Cell 29 — Model and training configuration

import torch
import torch.nn as nn
import timm
from pathlib import Path

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Hardware
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Model configuration
MODEL_NAME = "vit_base_patch16_224"
NUM_CLASSES = 3

# Fresh ImageNet-pretrained ViT-B/16
model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=NUM_CLASSES
)

model = model.to(DEVICE)

# Training hyperparameters
EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 3

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Mixed precision for RTX GPU
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

# Separate V4 output directories
MODEL_DIR = ROOT / "models" / "vit_v4"
RESULTS_DIR = ROOT / "results" / "vit_v4"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "vit_v4_best.pth"

print("V4 TRAINING CONFIGURATION")
print("-" * 45)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("Model:", MODEL_NAME)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Classes:", NUM_CLASSES)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Mixed precision:", scaler.is_enabled())
print("Checkpoint path:", BEST_MODEL_PATH)

print("\nPASS — Fresh ViT-B/16 initialized for V4.")


### Train V4 ViT-B/16


In [ ]:
# STEP 29 — Train V4 ViT-B/16
# STEP 30 — V4 ViT-B/16 training

import json
import time
import pandas as pd
import torch

# Verify training functions are available
assert callable(train_one_epoch)
assert callable(validate_one_epoch)

# Save experiment configuration
experiment_config = {
    "experiment": "vit_v4",
    "model": MODEL_NAME,
    "dataset_version": "v4",
    "num_classes": NUM_CLASSES,
    "class_names": ["normal", "immature", "mature"],
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "patience": PATIENCE,
    "seed": SEED,
    "optimizer": "AdamW",
    "loss": "CrossEntropyLoss",
    "mixed_precision": scaler.is_enabled(),
    "selection_metric": "validation_macro_f1"
}

with open(
    RESULTS_DIR / "experiment_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(experiment_config, f, indent=4)

# Training state
history = []
best_val_f1 = -1.0
epochs_without_improvement = 0

print("Starting ViT-B/16 V4 training...")
print("Device:", DEVICE)
print("Checkpoint:", BEST_MODEL_PATH)

for epoch in range(1, EPOCHS + 1):

    start_time = time.time()

    # Training
    train_loss, train_acc, train_f1 = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        scaler,
        DEVICE
    )

    # Validation
    val_loss, val_acc, val_f1 = validate_one_epoch(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    epoch_time = time.time() - start_time

    history.append({
        "epoch": epoch,
        "train_loss": float(train_loss),
        "train_accuracy": float(train_acc),
        "train_macro_f1": float(train_f1),
        "val_loss": float(val_loss),
        "val_accuracy": float(val_acc),
        "val_macro_f1": float(val_f1),
        "epoch_time_seconds": round(epoch_time, 2)
    })

    # Save training history after every epoch
    pd.DataFrame(history).to_csv(
        RESULTS_DIR / "training_history.csv",
        index=False
    )

    print(
        f"\nEpoch {epoch}/{EPOCHS} | "
        f"Time: {epoch_time:.1f}s\n"
        f"Train: Loss={train_loss:.4f}, "
        f"Acc={train_acc:.4f}, F1={train_f1:.4f}\n"
        f"Val:   Loss={val_loss:.4f}, "
        f"Acc={val_acc:.4f}, F1={val_f1:.4f}"
    )

    # Select best checkpoint using validation macro-F1
    if val_f1 > best_val_f1 + 1e-6:

        best_val_f1 = float(val_f1)
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print("Best V4 checkpoint saved.")

    else:
        epochs_without_improvement += 1

        print(
            "No validation F1 improvement:",
            f"{epochs_without_improvement}/{PATIENCE}"
        )

    if epochs_without_improvement >= PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\nV4 TRAINING COMPLETE")
print("-" * 45)
print("Epochs completed:", len(history))
print("Best validation macro-F1:", round(best_val_f1, 6))
print("Best checkpoint:", BEST_MODEL_PATH)
print("History:", RESULTS_DIR / "training_history.csv")


### Validate ViT predictions


In [ ]:
# STEP 30 — Validate ViT predictions
# STEP 31 — Independently verify V4 validation predictions

import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Load the best V4 checkpoint, not the final epoch weights
model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE,
        weights_only=True
    )
)

model.eval()

all_true = []
all_pred = []
all_confidences = []

with torch.inference_mode():
    for images, labels in val_loader:

        images = images.to(DEVICE)

        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)

        predictions = probabilities.argmax(dim=1)
        confidences = probabilities.max(dim=1).values

        all_true.extend(labels.cpu().numpy())
        all_pred.extend(predictions.cpu().numpy())
        all_confidences.extend(confidences.cpu().numpy())

all_true = np.array(all_true)
all_pred = np.array(all_pred)
all_confidences = np.array(all_confidences)

print("V4 VALIDATION VERIFICATION")
print("-" * 45)

print("Total predictions:", len(all_pred))
print("Correct predictions:", int((all_true == all_pred).sum()))
print("Incorrect predictions:", int((all_true != all_pred).sum()))

print("\nAccuracy:", accuracy_score(all_true, all_pred))
print("Macro-F1:", f1_score(all_true, all_pred, average="macro"))

print("\nConfusion matrix:")
print(confusion_matrix(all_true, all_pred, labels=[0, 1, 2]))

print("\nClassification report:")
print(
    classification_report(
        all_true,
        all_pred,
        target_names=["normal", "immature", "mature"],
        labels=[0, 1, 2],
        zero_division=0
    )
)

print("Mean confidence:", round(float(all_confidences.mean()), 6))
print("Minimum confidence:", round(float(all_confidences.min()), 6))

assert len(all_pred) == len(val_dataset)
assert len(all_true) == len(val_df)

print("\nPASS — Independent validation inference completed.")


### Audit class distribution by source


In [ ]:
# STEP 31 — Audit class distribution by source
# STEP 32 — Audit class distribution across dataset sources

import pandas as pd

# Load the full V4 manifest
audit_df = pd.read_csv(
    V4_DIR / "netrascan_3class_master_v4.csv"
)

print("SOURCE VS CLASS DISTRIBUTION")
print("-" * 55)

source_class_counts = pd.crosstab(
    audit_df["source"],
    audit_df["class"]
)

print("\nImage counts:")
print(source_class_counts)

print("\nClass percentages within each source:")
print(
    pd.crosstab(
        audit_df["source"],
        audit_df["class"],
        normalize="index"
    ).mul(100).round(2)
)

print("\nSOURCE VS SPLIT")
print("-" * 55)

print(
    pd.crosstab(
        audit_df["source"],
        audit_df["split"]
    )
)

print("\nSOURCE + CLASS VS SPLIT")
print("-" * 55)

print(
    pd.crosstab(
        [audit_df["source"], audit_df["class"]],
        audit_df["split"]
    )
)

print("\nPASS — Source distribution audit completed.")


### Inspect augmentation filenames


In [ ]:
# STEP 32 — Inspect augmentation filenames
# STEP 33 — Investigate Dataset Katarak augmentation filenames

import pandas as pd
import re

katarak_df = audit_df[
    audit_df["source"] == "dataset_katarak"
].copy()

# Remove the Roboflow export hash suffix
katarak_df["base_filename"] = (
    katarak_df["filename"]
    .str.replace(
        r"\.rf\.[0-9a-fA-F]+(?=\.[^.]+$)",
        "",
        regex=True
    )
    .str.lower()
)

# Extract common filename patterns
def get_filename_pattern(filename):
    pattern = re.sub(r"\d+", "<N>", filename)
    return pattern

katarak_df["filename_pattern"] = (
    katarak_df["base_filename"]
    .apply(get_filename_pattern)
)

print("DATASET KATARAK FILENAME AUDIT")
print("-" * 55)

print("Total images:", len(katarak_df))
print("Unique base filenames:", katarak_df["base_filename"].nunique())

print("\nMost common filename patterns:")
print(
    katarak_df["filename_pattern"]
    .value_counts()
    .head(15)
)

print("\nSample filenames:")
print(
    katarak_df[
        ["filename", "base_filename", "class", "split"]
    ].head(20).to_string(index=False)
)

# Check whether identical base filenames span splits
base_overlap = (
    katarak_df.groupby(
        ["class", "base_filename"]
    )["split"].nunique()
)

print(
    "\nBase filenames crossing splits:",
    int((base_overlap > 1).sum())
)

# Inspect largest V4 families from this source
print("\nLargest Katarak visual families:")
print(
    katarak_df.groupby("visual_family_id")
    .agg(
        images=("filename", "size"),
        classes=("class", "nunique"),
        splits=("split", "nunique")
    )
    .sort_values("images", ascending=False)
    .head(10)
)

print("\nPASS — Katarak filename audit completed.")


### Evaluate ViT on V4 test set


In [ ]:
# STEP 33 — Evaluate ViT on V4 test set
# STEP 34 — Evaluate V4 ViT-B/16 on the held-out test set

import torch
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Load the best checkpoint selected using validation macro-F1
model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE,
        weights_only=True
    )
)

model.eval()

test_true = []
test_pred = []
test_confidence = []

# Run inference on the held-out test dataset
with torch.inference_mode():
    for images, labels in test_loader:

        images = images.to(DEVICE)

        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)

        predictions = probabilities.argmax(dim=1)
        confidence = probabilities.max(dim=1).values

        test_true.extend(labels.cpu().numpy())
        test_pred.extend(predictions.cpu().numpy())
        test_confidence.extend(confidence.cpu().numpy())

test_true = np.array(test_true)
test_pred = np.array(test_pred)
test_confidence = np.array(test_confidence)

CLASS_NAMES = ["normal", "immature", "mature"]

# Calculate overall metrics
metrics = {
    "model": "ViT-B/16",
    "dataset": "V4",
    "test_images": len(test_true),
    "accuracy": accuracy_score(test_true, test_pred),
    "macro_precision": precision_score(
        test_true, test_pred, average="macro", zero_division=0
    ),
    "macro_recall": recall_score(
        test_true, test_pred, average="macro", zero_division=0
    ),
    "macro_f1": f1_score(
        test_true, test_pred, average="macro", zero_division=0
    ),
    "weighted_f1": f1_score(
        test_true, test_pred, average="weighted", zero_division=0
    )
}

print("V4 ViT-B/16 TEST EVALUATION")
print("-" * 50)

for key, value in metrics.items():
    print(f"{key}: {value}")

print("\nConfusion matrix:")
cm = confusion_matrix(test_true, test_pred, labels=[0, 1, 2])
print(cm)

print("\nClassification report:")
print(
    classification_report(
        test_true,
        test_pred,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES,
        zero_division=0
    )
)

print("Mean confidence:", round(float(test_confidence.mean()), 6))
print("Minimum confidence:", round(float(test_confidence.min()), 6))

# Save metrics and predictions
pd.DataFrame([metrics]).to_csv(
    RESULTS_DIR / "test_metrics.csv",
    index=False
)

prediction_df = test_df.copy()
prediction_df["true_label"] = test_true
prediction_df["predicted_label"] = test_pred
prediction_df["confidence"] = test_confidence
prediction_df["correct"] = test_true == test_pred

prediction_df.to_csv(
    RESULTS_DIR / "test_predictions.csv",
    index=False
)

assert len(test_true) == 571

print("\nPASS — Test evaluation completed and results saved.")


### Inspect ViT misclassifications


In [ ]:
# STEP 34 — Inspect ViT misclassifications
# STEP 35 — Inspect ViT-B/16 V4 test misclassifications

import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import math

# Load the saved test predictions
predictions_df = pd.read_csv(
    RESULTS_DIR / "test_predictions.csv"
)

errors_df = predictions_df[
    predictions_df["correct"] == False
].copy()

print("Total misclassified images:", len(errors_df))

print("\nErrors by source:")
print(errors_df["source"].value_counts())

print("\nErrors by actual class:")
print(errors_df["class"].value_counts())

# Display all misclassified images
n = len(errors_df)
cols = 5
rows = math.ceil(n / cols)

if n > 0:
    fig, axes = plt.subplots(
        rows, cols,
        figsize=(16, rows * 3.5)
    )

    axes = axes.flatten()

    for ax, (_, row) in zip(axes, errors_df.iterrows()):
        image = Image.open(row["image_path"]).convert("RGB")

        ax.imshow(image)
        ax.set_title(
            f"Actual: {CLASS_NAMES[int(row['true_label'])]}\n"
            f"Pred: {CLASS_NAMES[int(row['predicted_label'])]}\n"
            f"Confidence: {row['confidence']:.3f}",
            fontsize=9
        )
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    plt.suptitle(
        "ViT-B/16 — V4 Test Misclassifications",
        fontsize=15
    )

    plt.tight_layout()
    plt.show()
else:
    print("No misclassifications found.")

print("\nPASS — Misclassification inspection complete.")


### Analyze ViT errors by family


In [ ]:
# STEP 35 — Analyze ViT errors by family
# STEP 36 — Analyze ViT-B/16 errors by V4 visual family

import pandas as pd

errors_df = predictions_df[
    predictions_df["correct"] == False
].copy()

print("V4 MISCLASSIFICATION FAMILY ANALYSIS")
print("-" * 55)

print("Total misclassified images:", len(errors_df))

print(
    "Distinct misclassified visual families:",
    errors_df["visual_family_id"].nunique()
)

print("\nErrors grouped by visual family:")

family_errors = (
    errors_df.groupby(
        ["visual_family_id", "source", "class"]
    )
    .agg(
        error_count=("filename", "size"),
        mean_confidence=("confidence", "mean"),
        min_confidence=("confidence", "min")
    )
    .sort_values("error_count", ascending=False)
)

print(family_errors.to_string())

print("\nSource-specific test performance:")

source_performance = (
    predictions_df.groupby("source")
    .agg(
        total_images=("correct", "size"),
        correct_predictions=("correct", "sum")
    )
)

source_performance["errors"] = (
    source_performance["total_images"]
    - source_performance["correct_predictions"]
)

source_performance["accuracy"] = (
    source_performance["correct_predictions"]
    / source_performance["total_images"]
)

print(source_performance.to_string())

print("\nPASS — Error-family analysis completed.")


### Save ViT performance figures


In [ ]:
# STEP 36 — Save ViT performance figures
# STEP 37 — Generate and save ViT-B/16 V4 performance figures

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

history_df = pd.read_csv(
    RESULTS_DIR / "training_history.csv"
)

# Figure 1 — Training and validation accuracy
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    history_df["epoch"],
    history_df["train_accuracy"],
    marker="o",
    label="Training accuracy"
)

ax.plot(
    history_df["epoch"],
    history_df["val_accuracy"],
    marker="s",
    label="Validation accuracy"
)

ax.set_title("ViT-B/16 — V4 Accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.05)
ax.set_xticks(history_df["epoch"])
ax.legend()
ax.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(
    RESULTS_DIR / "vit_v4_accuracy_curve.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# Figure 2 — Held-out test confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Normal", "Immature", "Mature"]
)

display.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

ax.set_title("ViT-B/16 — V4 Test Confusion Matrix")

fig.tight_layout()
fig.savefig(
    RESULTS_DIR / "vit_v4_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("PASS — ViT-B/16 figures saved.")
print("Location:", RESULTS_DIR.resolve())


## 05 · Swin-Tiny: fine-tuning and evaluation

Swin-Tiny uses hierarchical shifted-window attention. Reported held-out V4 test accuracy: **98.95% (565/571)**.


### Restore V4 manifests for Swin experiment


In [ ]:
# STEP 37 — Restore V4 manifests for Swin experiment
# STEP 51A — Restore V4 dataset manifests

from pathlib import Path
import pandas as pd
import torch
import timm

ROOT = Path(r"C:\Personal AI & ML PROJECTS\AI IN HEALTHCARE\MY PROJECT E PAARVAI")

V4_DIR = ROOT / "NetraScan_Metadata" / "v4"

train_df = pd.read_csv(V4_DIR / "netrascan_train_v4.csv")
val_df = pd.read_csv(V4_DIR / "netrascan_val_v4.csv")
test_df = pd.read_csv(V4_DIR / "netrascan_test_v4.csv")

CLASS_NAMES = ["normal", "immature", "mature"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("V4 DATASET RESTORED")
print("-" * 45)
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Testing images:", len(test_df))
print("Classes:", CLASS_TO_IDX)
print("Device:", DEVICE)

assert len(train_df) == 2454
assert len(val_df) == 517
assert len(test_df) == 571

print("\nPASS — V4 manifests loaded successfully.")


### Recreate V4 data loaders


In [ ]:
# STEP 38 — Recreate V4 data loaders
# STEP 51 B — Restore V4 transforms and DataLoaders

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMAGE_SIZE = 224
BATCH_SIZE = 16

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(8),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

test_transform = val_transform

class CataractDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        label = CLASS_TO_IDX[row["class"]]

        if self.transform:
            image = self.transform(image)

        return image, label

train_dataset = CataractDataset(train_df, train_transform)
val_dataset = CataractDataset(val_df, val_transform)
test_dataset = CataractDataset(test_df, test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

images, labels = next(iter(train_loader))

print("V4 DATALOADERS RESTORED")
print("-" * 45)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

print("\nPASS — V4 DataLoaders ready.")


### Initialize Swin-Tiny


In [ ]:
# STEP 39 — Initialize Swin-Tiny
# STEP 52 — Swin-Tiny initialization

import timm
import torch

SWIN_MODEL_NAME = "swin_tiny_patch4_window7_224"

swin_model = timm.create_model(
    SWIN_MODEL_NAME,
    pretrained=True,
    num_classes=3
).to(DEVICE)

swin_parameters = sum(
    p.numel() for p in swin_model.parameters()
)

swin_config = timm.data.resolve_model_data_config(swin_model)

print("SWIN-TINY INITIALIZATION")
print("-" * 45)
print("Architecture:", SWIN_MODEL_NAME)
print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")
print("Parameters:", f"{swin_parameters:,}")
print("Output classes:", swin_model.num_classes)
print("Input size:", swin_config["input_size"])
print("Pretrained mean:", swin_config["mean"])
print("Pretrained std:", swin_config["std"])

swin_model.eval()

with torch.inference_mode():
    sample_images, _ = next(iter(val_loader))
    outputs = swin_model(sample_images[:2].to(DEVICE))

print("Output shape:", tuple(outputs.shape))

assert outputs.shape == (2, 3)

print("\nPASS — Swin-Tiny initialized successfully.")


### Configure Swin preprocessing


In [ ]:
# STEP 40 — Configure Swin preprocessing
# STEP 53 — Swin-Tiny preprocessing and DataLoaders

from torchvision import transforms
from torch.utils.data import DataLoader

SWIN_MEAN = swin_config["mean"]
SWIN_STD = swin_config["std"]

swin_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(8),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(SWIN_MEAN, SWIN_STD)
])

swin_eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(SWIN_MEAN, SWIN_STD)
])

swin_train_dataset = CataractDataset(
    train_df, swin_train_transform
)

swin_val_dataset = CataractDataset(
    val_df, swin_eval_transform
)

swin_test_dataset = CataractDataset(
    test_df, swin_eval_transform
)

swin_train_loader = DataLoader(
    swin_train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

swin_val_loader = DataLoader(
    swin_val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

swin_test_loader = DataLoader(
    swin_test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

images, labels = next(iter(swin_train_loader))

print("SWIN-TINY V4 DATALOADERS")
print("-" * 45)
print("Training images:", len(swin_train_dataset))
print("Validation images:", len(swin_val_dataset))
print("Testing images:", len(swin_test_dataset))
print("Training batches:", len(swin_train_loader))
print("Validation batches:", len(swin_val_loader))
print("Testing batches:", len(swin_test_loader))
print("Batch shape:", images.shape)
print("Normalization mean:", SWIN_MEAN)
print("Normalization std:", SWIN_STD)

assert images.shape == (16, 3, 224, 224)

print("\nPASS — Swin-Tiny preprocessing ready.")


### Configure Swin training


In [ ]:
# STEP 41 — Configure Swin training
# STEP 54 — Swin-Tiny V4 training configuration

import json
import torch
import torch.nn as nn
from pathlib import Path
from torch.amp import GradScaler

SWIN_EPOCHS = 10
SWIN_LR = 2e-5
SWIN_WEIGHT_DECAY = 0.01
SWIN_PATIENCE = 3

SWIN_MODEL_DIR = ROOT / "models" / "swin_v4"
SWIN_RESULTS_DIR = ROOT / "results" / "swin_v4"

SWIN_MODEL_DIR.mkdir(parents=True, exist_ok=True)
SWIN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SWIN_CHECKPOINT = SWIN_MODEL_DIR / "swin_v4_best.pth"

swin_criterion = nn.CrossEntropyLoss()

swin_optimizer = torch.optim.AdamW(
    swin_model.parameters(),
    lr=SWIN_LR,
    weight_decay=SWIN_WEIGHT_DECAY
)

swin_scaler = GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

swin_experiment_config = {
    "model": "Swin-Tiny",
    "architecture": SWIN_MODEL_NAME,
    "dataset": "V4",
    "train_images": len(swin_train_dataset),
    "val_images": len(swin_val_dataset),
    "test_images": len(swin_test_dataset),
    "num_classes": 3,
    "epochs": SWIN_EPOCHS,
    "batch_size": 16,
    "learning_rate": SWIN_LR,
    "weight_decay": SWIN_WEIGHT_DECAY,
    "early_stopping_patience": SWIN_PATIENCE,
    "optimizer": "AdamW",
    "loss": "CrossEntropyLoss",
    "mixed_precision": DEVICE.type == "cuda",
    "normalization_mean": list(SWIN_MEAN),
    "normalization_std": list(SWIN_STD),
    "parameters": swin_parameters
}

with open(
    SWIN_RESULTS_DIR / "experiment_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(swin_experiment_config, f, indent=4)

print("SWIN-TINY V4 TRAINING CONFIGURATION")
print("-" * 50)
print("Device:", DEVICE)
print("Architecture:", SWIN_MODEL_NAME)
print("Parameters:", f"{swin_parameters:,}")
print("Epochs:", SWIN_EPOCHS)
print("Learning rate:", SWIN_LR)
print("Batch size:", 16)
print("Mixed precision:", DEVICE.type == "cuda")
print("Checkpoint:", SWIN_CHECKPOINT)
print("Results directory:", SWIN_RESULTS_DIR)

print("\nPASS — Swin-Tiny training configuration ready.")


### Train Swin-Tiny


In [ ]:
# STEP 42 — Train Swin-Tiny
# STEP 55 — Swin-Tiny V4 training

import time
import pandas as pd
import numpy as np
import torch
from torch.amp import autocast
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

def swin_run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    all_labels = []
    all_predictions = []

    for images, labels in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda")
            ):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * images.size(0)
        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(outputs.argmax(dim=1).detach().cpu().numpy())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_predictions),
        "macro_f1": f1_score(
            all_labels,
            all_predictions,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0
        )
    }

swin_history = []
swin_best_f1 = -1.0
swin_no_improvement = 0

print("Starting Swin-Tiny V4 training...")
print("Device:", DEVICE)
print("Checkpoint:", SWIN_CHECKPOINT)

for epoch in range(1, SWIN_EPOCHS + 1):
    start_time = time.time()

    train_metrics = swin_run_epoch(
        swin_model,
        swin_train_loader,
        swin_criterion,
        swin_optimizer,
        swin_scaler
    )

    val_metrics = swin_run_epoch(
        swin_model,
        swin_val_loader,
        swin_criterion
    )

    elapsed = time.time() - start_time

    swin_history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "epoch_seconds": elapsed
    })

    print(f"\nEpoch {epoch}/{SWIN_EPOCHS} | Time: {elapsed:.1f}s")
    print(
        f"Train: Loss={train_metrics['loss']:.4f}, "
        f"Acc={train_metrics['accuracy']:.4f}, "
        f"F1={train_metrics['macro_f1']:.4f}"
    )
    print(
        f"Val:   Loss={val_metrics['loss']:.4f}, "
        f"Acc={val_metrics['accuracy']:.4f}, "
        f"F1={val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["macro_f1"] > swin_best_f1 + 1e-8:
        swin_best_f1 = val_metrics["macro_f1"]
        swin_no_improvement = 0

        torch.save(
            swin_model.state_dict(),
            SWIN_CHECKPOINT
        )

        print("Best Swin-Tiny checkpoint saved.")
    else:
        swin_no_improvement += 1
        print(
            f"No validation F1 improvement: "
            f"{swin_no_improvement}/{SWIN_PATIENCE}"
        )

    pd.DataFrame(swin_history).to_csv(
        SWIN_RESULTS_DIR / "training_history.csv",
        index=False
    )

    if swin_no_improvement >= SWIN_PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\nSWIN-TINY V4 TRAINING COMPLETE")
print("-" * 45)
print("Epochs completed:", len(swin_history))
print("Best validation macro-F1:", round(swin_best_f1, 6))
print("Best checkpoint:", SWIN_CHECKPOINT)
print("History:", SWIN_RESULTS_DIR / "training_history.csv")


### Evaluate Swin-Tiny


In [ ]:
# STEP 43 — Evaluate Swin-Tiny
# STEP 56 — Swin-Tiny V4 test evaluation

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

swin_model.load_state_dict(
    torch.load(SWIN_CHECKPOINT, map_location=DEVICE, weights_only=True)
)

swin_model.eval()

swin_true = []
swin_pred = []
swin_confidence = []

with torch.inference_mode():
    for images, labels in swin_test_loader:
        images = images.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            outputs = swin_model(images)

        probabilities = torch.softmax(outputs.float(), dim=1)
        confidence, predictions = probabilities.max(dim=1)

        swin_true.extend(labels.numpy().tolist())
        swin_pred.extend(predictions.cpu().numpy().tolist())
        swin_confidence.extend(confidence.cpu().numpy().tolist())

swin_accuracy = accuracy_score(swin_true, swin_pred)
swin_precision = precision_score(
    swin_true, swin_pred, average="macro", zero_division=0
)
swin_recall = recall_score(
    swin_true, swin_pred, average="macro", zero_division=0
)
swin_f1 = f1_score(
    swin_true, swin_pred, average="macro", zero_division=0
)

swin_cm = confusion_matrix(swin_true, swin_pred, labels=[0, 1, 2])

swin_metrics = pd.DataFrame([{
    "model": "Swin-Tiny",
    "dataset": "V4",
    "test_images": len(swin_true),
    "accuracy": swin_accuracy,
    "macro_precision": swin_precision,
    "macro_recall": swin_recall,
    "macro_f1": swin_f1,
    "parameters": swin_parameters
}])

swin_predictions = test_df.copy()
swin_predictions["true_label"] = [
    CLASS_NAMES[i] for i in swin_true
]
swin_predictions["predicted_label"] = [
    CLASS_NAMES[i] for i in swin_pred
]
swin_predictions["confidence"] = swin_confidence
swin_predictions["correct"] = np.array(swin_true) == np.array(swin_pred)

swin_metrics.to_csv(
    SWIN_RESULTS_DIR / "test_metrics.csv", index=False
)

swin_predictions.to_csv(
    SWIN_RESULTS_DIR / "test_predictions.csv", index=False
)

print("SWIN-TINY V4 TEST EVALUATION")
print("-" * 50)
print(swin_metrics.to_string(index=False))
print("\nConfusion matrix:")
print(swin_cm)
print("\nClassification report:")
print(
    classification_report(
        swin_true,
        swin_pred,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES,
        zero_division=0
    )
)
print("Mean confidence:", round(float(np.mean(swin_confidence)), 6))
print("Minimum confidence:", round(float(np.min(swin_confidence)), 6))
print("\nPASS — Swin-Tiny test evaluation completed.")


### Analyze Swin errors


In [ ]:
# STEP 44 — Analyze Swin errors
# STEP 57 — Swin-Tiny V4 misclassification analysis

swin_errors = swin_predictions[
    ~swin_predictions["correct"]
].copy()

print("SWIN-TINY V4 ERROR ANALYSIS")
print("-" * 50)

print("Total misclassified images:", len(swin_errors))
print("Distinct visual families:", swin_errors["visual_family_id"].nunique())

print("\nErrors by source:")
print(swin_errors["source"].value_counts())

print("\nErrors by actual class:")
print(swin_errors["true_label"].value_counts())

print("\nErrors grouped by visual family:")
print(
    swin_errors.groupby(
        ["visual_family_id", "source", "true_label"]
    ).agg(
        error_count=("correct", "size"),
        mean_confidence=("confidence", "mean"),
        min_confidence=("confidence", "min")
    ).sort_values("error_count", ascending=False)
)

print("\nSource-specific test performance:")
print(
    swin_predictions.groupby("source").agg(
        total_images=("correct", "size"),
        correct_predictions=("correct", "sum"),
        accuracy=("correct", "mean")
    )
)

print("\nPASS — Swin-Tiny error analysis completed.")


### Save Swin figures


In [ ]:
# STEP 45 — Save Swin figures
# STEP 58 — Swin-Tiny V4 performance visualizations

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay

swin_history_df = pd.read_csv(
    SWIN_RESULTS_DIR / "training_history.csv"
)

plt.figure(figsize=(9, 5))

plt.plot(
    swin_history_df["epoch"],
    swin_history_df["train_accuracy"],
    marker="o",
    label="Training accuracy"
)

plt.plot(
    swin_history_df["epoch"],
    swin_history_df["val_accuracy"],
    marker="s",
    label="Validation accuracy"
)

plt.title("Swin-Tiny — V4 Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

plt.savefig(
    SWIN_RESULTS_DIR / "swin_v4_accuracy_curve.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

fig, ax = plt.subplots(figsize=(7, 6))

display = ConfusionMatrixDisplay(
    confusion_matrix=swin_cm,
    display_labels=["Normal", "Immature", "Mature"]
)

display.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

plt.title("Swin-Tiny — V4 Test Confusion Matrix")
plt.tight_layout()

plt.savefig(
    SWIN_RESULTS_DIR / "swin_v4_confusion_matrix.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("PASS — Swin-Tiny figures saved.")
print("Location:", SWIN_RESULTS_DIR)


## 06 · MobileViT-S: fine-tuning and evaluation

MobileViT-S combines convolutional processing with transformer components. Reported held-out V4 test accuracy: **98.25% (561/571)**.


### Initialize MobileViT-S


In [ ]:
# STEP 46 — Initialize MobileViT-S
# STEP 59 — MobileViT-S V4 model initialization

import timm
import torch

MOBILEVIT_MODEL_NAME = "mobilevit_s"

mobilevit_model = timm.create_model(
    MOBILEVIT_MODEL_NAME,
    pretrained=True,
    num_classes=3
).to(DEVICE)

mobilevit_config = timm.data.resolve_model_data_config(
    mobilevit_model
)

mobilevit_parameters = sum(
    p.numel() for p in mobilevit_model.parameters()
)

mobilevit_model.eval()

with torch.inference_mode():
    sample_input = torch.randn(
        2, 3, 256, 256,
        device=DEVICE
    )
    sample_output = mobilevit_model(sample_input)

print("MOBILEVIT-S INITIALIZATION")
print("-" * 50)
print("Architecture:", MOBILEVIT_MODEL_NAME)
print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))
print("Parameters:", f"{mobilevit_parameters:,}")
print("Output classes:", 3)
print("Pretrained input size:", mobilevit_config["input_size"])
print("Pretrained mean:", mobilevit_config["mean"])
print("Pretrained std:", mobilevit_config["std"])
print("Output shape:", tuple(sample_output.shape))

assert sample_output.shape == (2, 3)

print("\nPASS — MobileViT-S initialized successfully.")


### Configure MobileViT preprocessing


In [ ]:
# STEP 47 — Configure MobileViT preprocessing
# STEP 60 — MobileViT-S V4 preprocessing and DataLoaders

from torchvision import transforms
from torch.utils.data import DataLoader

MOBILEVIT_SIZE = mobilevit_config["input_size"][-2:]
MOBILEVIT_MEAN = mobilevit_config["mean"]
MOBILEVIT_STD = mobilevit_config["std"]

mobilevit_train_transform = transforms.Compose([
    transforms.Resize(MOBILEVIT_SIZE),
    transforms.RandomRotation(8),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(MOBILEVIT_MEAN, MOBILEVIT_STD)
])

mobilevit_eval_transform = transforms.Compose([
    transforms.Resize(MOBILEVIT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MOBILEVIT_MEAN, MOBILEVIT_STD)
])

mobilevit_train_dataset = CataractDataset(
    train_df, mobilevit_train_transform
)
mobilevit_val_dataset = CataractDataset(
    val_df, mobilevit_eval_transform
)
mobilevit_test_dataset = CataractDataset(
    test_df, mobilevit_eval_transform
)

mobilevit_train_loader = DataLoader(
    mobilevit_train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

mobilevit_val_loader = DataLoader(
    mobilevit_val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

mobilevit_test_loader = DataLoader(
    mobilevit_test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

images, labels = next(iter(mobilevit_train_loader))

print("MOBILEVIT-S V4 DATALOADERS")
print("-" * 45)
print("Training images:", len(mobilevit_train_dataset))
print("Validation images:", len(mobilevit_val_dataset))
print("Testing images:", len(mobilevit_test_dataset))
print("Training batches:", len(mobilevit_train_loader))
print("Validation batches:", len(mobilevit_val_loader))
print("Testing batches:", len(mobilevit_test_loader))
print("Batch shape:", images.shape)
print("Normalization mean:", MOBILEVIT_MEAN)
print("Normalization std:", MOBILEVIT_STD)

assert images.shape[1:] == (3, 256, 256)
assert len(mobilevit_train_dataset) == 2454
assert len(mobilevit_val_dataset) == 517
assert len(mobilevit_test_dataset) == 571

print("\nPASS — MobileViT-S V4 DataLoaders ready.")


### Configure MobileViT training


In [ ]:
# STEP 48 — Configure MobileViT training
# STEP 61 — MobileViT-S V4 training configuration

import torch
import torch.nn as nn
from pathlib import Path

MOBILEVIT_EPOCHS = 10
MOBILEVIT_LR = 2e-5
MOBILEVIT_WEIGHT_DECAY = 0.01
MOBILEVIT_PATIENCE = 3

MOBILEVIT_MODEL_DIR = ROOT / "models" / "mobilevit_v4"
MOBILEVIT_RESULTS_DIR = ROOT / "results" / "mobilevit_v4"

MOBILEVIT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
MOBILEVIT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MOBILEVIT_CHECKPOINT = (
    MOBILEVIT_MODEL_DIR / "mobilevit_v4_best.pth"
)

mobilevit_model.train()

mobilevit_criterion = nn.CrossEntropyLoss()

mobilevit_optimizer = torch.optim.AdamW(
    mobilevit_model.parameters(),
    lr=MOBILEVIT_LR,
    weight_decay=MOBILEVIT_WEIGHT_DECAY
)

mobilevit_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

print("MOBILEVIT-S V4 TRAINING CONFIGURATION")
print("-" * 50)
print("Device:", DEVICE)
print("Architecture:", MOBILEVIT_MODEL_NAME)
print("Parameters:", f"{mobilevit_parameters:,}")
print("Epochs:", MOBILEVIT_EPOCHS)
print("Learning rate:", MOBILEVIT_LR)
print("Batch size:", mobilevit_train_loader.batch_size)
print("Weight decay:", MOBILEVIT_WEIGHT_DECAY)
print("Mixed precision:", mobilevit_scaler.is_enabled())
print("Early stopping patience:", MOBILEVIT_PATIENCE)
print("Checkpoint:", MOBILEVIT_CHECKPOINT)
print("Results directory:", MOBILEVIT_RESULTS_DIR)

print("\nPASS — MobileViT-S training configuration ready.")


### Train MobileViT-S


In [ ]:
# STEP 49 — Train MobileViT-S
# STEP 62 — MobileViT-S V4 training

import time
import json
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

def mobilevit_run_epoch(model, loader, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    all_true = []
    all_pred = []

    for images, labels in tqdm(
        loader,
        desc="Train" if training else "Validation",
        leave=False
    ):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda")
            ):
                outputs = model(images)
                loss = mobilevit_criterion(outputs, labels)

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * labels.size(0)
        all_true.extend(labels.cpu().tolist())
        all_pred.extend(outputs.argmax(dim=1).cpu().tolist())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_true, all_pred),
        "macro_f1": f1_score(
            all_true,
            all_pred,
            average="macro",
            zero_division=0
        )
    }


mobilevit_history = []
mobilevit_best_f1 = -1.0
mobilevit_no_improvement = 0

mobilevit_experiment_config = {
    "model": "MobileViT-S",
    "architecture": MOBILEVIT_MODEL_NAME,
    "dataset": "V4",
    "train_images": len(mobilevit_train_dataset),
    "val_images": len(mobilevit_val_dataset),
    "test_images": len(mobilevit_test_dataset),
    "parameters": mobilevit_parameters,
    "epochs_max": MOBILEVIT_EPOCHS,
    "learning_rate": MOBILEVIT_LR,
    "weight_decay": MOBILEVIT_WEIGHT_DECAY,
    "batch_size": mobilevit_train_loader.batch_size,
    "patience": MOBILEVIT_PATIENCE,
    "checkpoint_selection": "validation_macro_f1",
    "input_size": list(MOBILEVIT_SIZE),
    "mean": list(MOBILEVIT_MEAN),
    "std": list(MOBILEVIT_STD)
}

with open(
    MOBILEVIT_RESULTS_DIR / "experiment_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(mobilevit_experiment_config, f, indent=2)

print("Starting MobileViT-S V4 training...")
print("Device:", DEVICE)
print("Checkpoint:", MOBILEVIT_CHECKPOINT)

for epoch in range(1, MOBILEVIT_EPOCHS + 1):
    start_time = time.time()

    train_metrics = mobilevit_run_epoch(
        mobilevit_model,
        mobilevit_train_loader,
        optimizer=mobilevit_optimizer,
        scaler=mobilevit_scaler
    )

    val_metrics = mobilevit_run_epoch(
        mobilevit_model,
        mobilevit_val_loader
    )

    elapsed = time.time() - start_time

    mobilevit_history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "epoch_seconds": elapsed
    })

    pd.DataFrame(mobilevit_history).to_csv(
        MOBILEVIT_RESULTS_DIR / "training_history.csv",
        index=False
    )

    print(f"\nEpoch {epoch}/{MOBILEVIT_EPOCHS} | Time: {elapsed:.1f}s")
    print(
        f"Train: Loss={train_metrics['loss']:.4f}, "
        f"Acc={train_metrics['accuracy']:.4f}, "
        f"F1={train_metrics['macro_f1']:.4f}"
    )
    print(
        f"Val:   Loss={val_metrics['loss']:.4f}, "
        f"Acc={val_metrics['accuracy']:.4f}, "
        f"F1={val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["macro_f1"] > mobilevit_best_f1 + 1e-6:
        mobilevit_best_f1 = val_metrics["macro_f1"]
        mobilevit_no_improvement = 0

        torch.save(
            mobilevit_model.state_dict(),
            MOBILEVIT_CHECKPOINT
        )

        print("Best MobileViT-S checkpoint saved.")
    else:
        mobilevit_no_improvement += 1
        print(
            "No validation F1 improvement:",
            f"{mobilevit_no_improvement}/{MOBILEVIT_PATIENCE}"
        )

    if mobilevit_no_improvement >= MOBILEVIT_PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\nMOBILEVIT-S V4 TRAINING COMPLETE")
print("-" * 45)
print("Epochs completed:", len(mobilevit_history))
print("Best validation macro-F1:", round(mobilevit_best_f1, 6))
print("Best checkpoint:", MOBILEVIT_CHECKPOINT)
print(
    "History:",
    MOBILEVIT_RESULTS_DIR / "training_history.csv"
)


### Evaluate MobileViT-S


In [ ]:
# STEP 50 — Evaluate MobileViT-S
# STEP 63 — MobileViT-S V4 test evaluation

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

mobilevit_model.load_state_dict(
    torch.load(
        MOBILEVIT_CHECKPOINT,
        map_location=DEVICE,
        weights_only=True
    )
)

mobilevit_model.eval()

mobilevit_true = []
mobilevit_pred = []
mobilevit_confidence = []

with torch.inference_mode():
    for images, labels in mobilevit_test_loader:
        images = images.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            outputs = mobilevit_model(images)

        probabilities = torch.softmax(outputs.float(), dim=1)
        confidence, predictions = probabilities.max(dim=1)

        mobilevit_true.extend(labels.tolist())
        mobilevit_pred.extend(predictions.cpu().tolist())
        mobilevit_confidence.extend(confidence.cpu().tolist())

mobilevit_accuracy = accuracy_score(
    mobilevit_true, mobilevit_pred
)

precision, recall, macro_f1, _ = (
    precision_recall_fscore_support(
        mobilevit_true,
        mobilevit_pred,
        average="macro",
        zero_division=0
    )
)

mobilevit_cm = confusion_matrix(
    mobilevit_true,
    mobilevit_pred,
    labels=[0, 1, 2]
)

mobilevit_metrics = pd.DataFrame([{
    "model": "MobileViT-S",
    "dataset": "V4",
    "test_images": len(mobilevit_true),
    "accuracy": mobilevit_accuracy,
    "macro_precision": precision,
    "macro_recall": recall,
    "macro_f1": macro_f1,
    "parameters": mobilevit_parameters
}])

mobilevit_metrics.to_csv(
    MOBILEVIT_RESULTS_DIR / "test_metrics.csv",
    index=False
)

mobilevit_predictions = test_df.reset_index(drop=True).copy()

mobilevit_predictions["true_label"] = [
    CLASS_NAMES[i] for i in mobilevit_true
]

mobilevit_predictions["predicted_label"] = [
    CLASS_NAMES[i] for i in mobilevit_pred
]

mobilevit_predictions["confidence"] = mobilevit_confidence

mobilevit_predictions["correct"] = (
    np.array(mobilevit_true) == np.array(mobilevit_pred)
)

mobilevit_predictions.to_csv(
    MOBILEVIT_RESULTS_DIR / "test_predictions.csv",
    index=False
)

print("MOBILEVIT-S V4 TEST EVALUATION")
print("-" * 50)

print(mobilevit_metrics.to_string(index=False))

print("\nConfusion matrix:")
print(mobilevit_cm)

print("\nClassification report:")
print(
    classification_report(
        mobilevit_true,
        mobilevit_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )
)

print("Mean confidence:", round(np.mean(mobilevit_confidence), 6))
print("Minimum confidence:", round(np.min(mobilevit_confidence), 6))

print("\nPASS — MobileViT-S test evaluation completed.")


### Analyze MobileViT errors


In [ ]:
# STEP 51 — Analyze MobileViT errors
# STEP 64 — MobileViT-S V4 misclassification analysis

mobilevit_errors = mobilevit_predictions[
    ~mobilevit_predictions["correct"]
].copy()

print("MOBILEVIT-S V4 ERROR ANALYSIS")
print("-" * 50)

print("Total misclassified images:", len(mobilevit_errors))
print(
    "Distinct visual families:",
    mobilevit_errors["visual_family_id"].nunique()
)

print("\nErrors by source:")
print(mobilevit_errors["source"].value_counts())

print("\nErrors by actual class:")
print(mobilevit_errors["true_label"].value_counts())

print("\nErrors grouped by visual family:")
print(
    mobilevit_errors.groupby(
        ["visual_family_id", "source", "true_label"]
    ).agg(
        error_count=("correct", "size"),
        mean_confidence=("confidence", "mean"),
        min_confidence=("confidence", "min")
    ).sort_values("error_count", ascending=False)
)

print("\nSource-specific test performance:")
print(
    mobilevit_predictions.groupby("source").agg(
        total_images=("correct", "size"),
        correct_predictions=("correct", "sum"),
        accuracy=("correct", "mean")
    )
)

print("\nPASS — MobileViT-S error analysis completed.")


### Save MobileViT figures


In [ ]:
# STEP 52 — Save MobileViT figures
# STEP 65 — MobileViT-S V4 performance visualizations

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

mobilevit_history_df = pd.read_csv(
    MOBILEVIT_RESULTS_DIR / "training_history.csv"
)

plt.figure(figsize=(9, 5))

plt.plot(
    mobilevit_history_df["epoch"],
    mobilevit_history_df["train_accuracy"],
    marker="o",
    label="Training accuracy"
)

plt.plot(
    mobilevit_history_df["epoch"],
    mobilevit_history_df["val_accuracy"],
    marker="s",
    label="Validation accuracy"
)

plt.title("MobileViT-S — V4 Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

plt.savefig(
    MOBILEVIT_RESULTS_DIR / "mobilevit_v4_accuracy_curve.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

fig, ax = plt.subplots(figsize=(7, 6))

display = ConfusionMatrixDisplay(
    confusion_matrix=mobilevit_cm,
    display_labels=["Normal", "Immature", "Mature"]
)

display.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

plt.title("MobileViT-S — V4 Test Confusion Matrix")
plt.tight_layout()

plt.savefig(
    MOBILEVIT_RESULTS_DIR / "mobilevit_v4_confusion_matrix.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("PASS — MobileViT-S figures saved.")
print("Location:", MOBILEVIT_RESULTS_DIR)


## 07 · Comparative evaluation and error analysis

Compare saved per-model metrics and prediction files. The corrected cross-model analysis normalizes label representations before joining predictions; the one-off diagnostic cell is omitted.


### Compare three models


In [ ]:
# STEP 53 — Compare three models
# STEP 66 — V4 three-model comparison 

import pandas as pd

model_configs = {
    "ViT-B/16": {
        "path": ROOT / "results" / "vit_v4" / "test_metrics.csv",
        "parameters": 85800963
    },
    "Swin-Tiny": {
        "path": ROOT / "results" / "swin_v4" / "test_metrics.csv",
        "parameters": 27521661
    },
    "MobileViT-S": {
        "path": ROOT / "results" / "mobilevit_v4" / "test_metrics.csv",
        "parameters": 4939555
    }
}

comparison_rows = []

for model_name, config in model_configs.items():
    metrics = pd.read_csv(config["path"]).iloc[0]

    comparison_rows.append({
        "model": model_name,
        "accuracy": float(metrics["accuracy"]),
        "macro_precision": float(metrics["macro_precision"]),
        "macro_recall": float(metrics["macro_recall"]),
        "macro_f1": float(metrics["macro_f1"]),
        "parameters": config["parameters"],
        "test_images": int(metrics["test_images"])
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df["correct_predictions"] = (
    comparison_df["accuracy"] * comparison_df["test_images"]
).round().astype(int)

comparison_df["errors"] = (
    comparison_df["test_images"] - comparison_df["correct_predictions"]
)

comparison_df = comparison_df.sort_values(
    "macro_f1", ascending=False
).reset_index(drop=True)

comparison_dir = ROOT / "results" / "comparison_v4"
comparison_dir.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(
    comparison_dir / "three_model_comparison.csv",
    index=False
)

print("NETRASCAN.AI — V4 MODEL COMPARISON")
print("-" * 75)
print(comparison_df.to_string(index=False))

print("\nBest macro-F1:", comparison_df.iloc[0]["model"])

print("Fewest parameters:",
      comparison_df.loc[
          comparison_df["parameters"].idxmin(), "model"
      ])

print("\nPASS — Three-model comparison saved.")



### Plot comparison dashboard


In [ ]:
# STEP 54 — Plot comparison dashboard
# STEP 67 — V4 Model Comparison Dashboard

import matplotlib.pyplot as plt
import numpy as np

plot_df = comparison_df.copy()

names = plot_df["model"].tolist()
accuracy = plot_df["accuracy"].to_numpy() * 100
macro_f1 = plot_df["macro_f1"].to_numpy() * 100
parameters = plot_df["parameters"].to_numpy() / 1_000_000
errors = plot_df["errors"].to_numpy()

colors = ["#4ECDC4", "#5B8DEF", "#F4A261"]

plt.style.use("dark_background")

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.patch.set_facecolor("#0B1220")

for ax in axes.flat:
    ax.set_facecolor("#151E2E")
    ax.grid(axis="y", alpha=0.15)
    ax.set_axisbelow(True)

bars = axes[0, 0].bar(names, accuracy, color=colors, width=0.55)
axes[0, 0].set_title("Test Accuracy", fontsize=15, weight="bold")
axes[0, 0].set_ylabel("Accuracy (%)")
axes[0, 0].set_ylim(0, 105)

for bar, value in zip(bars, accuracy):
    axes[0, 0].text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}%",
        ha="center",
        fontsize=11,
        weight="bold"
    )

bars = axes[0, 1].bar(names, macro_f1, color=colors, width=0.55)
axes[0, 1].set_title("Macro-F1 Score", fontsize=15, weight="bold")
axes[0, 1].set_ylabel("Macro-F1 (%)")
axes[0, 1].set_ylim(0, 105)

for bar, value in zip(bars, macro_f1):
    axes[0, 1].text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}%",
        ha="center",
        fontsize=11,
        weight="bold"
    )

bars = axes[1, 0].bar(names, parameters, color=colors, width=0.55)
axes[1, 0].set_title("Model Complexity", fontsize=15, weight="bold")
axes[1, 0].set_ylabel("Parameters (Millions)")
axes[1, 0].set_ylim(0, max(parameters) * 1.18)

for bar, value in zip(bars, parameters):
    axes[1, 0].text(
        bar.get_x() + bar.get_width() / 2,
        value + 1.5,
        f"{value:.2f}M",
        ha="center",
        fontsize=11,
        weight="bold"
    )

for i, name in enumerate(names):
    axes[1, 1].scatter(
        parameters[i],
        accuracy[i],
        s=220,
        color=colors[i],
        edgecolor="white",
        linewidth=1.2,
        label=name,
        zorder=3
    )

    axes[1, 1].annotate(
        name,
        (parameters[i], accuracy[i]),
        xytext=(8, 9),
        textcoords="offset points",
        fontsize=10
    )

axes[1, 1].set_title(
    "Accuracy vs Model Complexity",
    fontsize=15,
    weight="bold"
)
axes[1, 1].set_xlabel("Parameters (Millions)")
axes[1, 1].set_ylabel("Test Accuracy (%)")
axes[1, 1].set_xlim(0, max(parameters) * 1.2)
axes[1, 1].set_ylim(95, 100)
axes[1, 1].grid(alpha=0.2)
axes[1, 1].legend(loc="lower right")

fig.suptitle(
    "NetraScan.AI | Transformer Architecture Comparison",
    fontsize=22,
    weight="bold",
    y=0.98
)

fig.text(
    0.5,
    0.935,
    "V4 Dataset | 571 Test Images | Normal / Immature / Mature Cataract",
    ha="center",
    fontsize=12,
    color="#AAB7C4"
)

plt.tight_layout(rect=[0, 0.02, 1, 0.91])

dashboard_path = comparison_dir / "model_comparison_dashboard.png"

plt.savefig(
    dashboard_path,
    dpi=300,
    bbox_inches="tight",
    facecolor=fig.get_facecolor()
)

plt.show()

print("PASS — Model comparison dashboard saved.")
print("Location:", dashboard_path)


### Analyze cross-model errors


In [ ]:
# STEP 55 — Analyze cross-model errors
# STEP 68B — Corrected V4 Cross-Model Error Analysis

import pandas as pd
import numpy as np

LABEL_MAP = {
    "0": "normal",
    "1": "immature",
    "2": "mature",
    "normal": "normal",
    "immature": "immature",
    "mature": "mature"
}

prediction_files = {
    "ViT-B/16": ROOT / "results" / "vit_v4" / "test_predictions.csv",
    "Swin-Tiny": ROOT / "results" / "swin_v4" / "test_predictions.csv",
    "MobileViT-S": ROOT / "results" / "mobilevit_v4" / "test_predictions.csv"
}

def normalize_labels(series):
    normalized = series.astype(str).str.strip().str.lower().map(LABEL_MAP)
    if normalized.isna().any():
        raise ValueError(
            f"Unknown labels: {series[normalized.isna()].unique()}"
        )
    return normalized

model_predictions = {}

for model_name, path in prediction_files.items():
    df = pd.read_csv(path)

    df["image_key"] = (
        df["source"].astype(str) + "::" +
        df["filename"].astype(str)
    )

    if df["image_key"].duplicated().any():
        raise ValueError(f"Duplicate image keys: {model_name}")

    df["true_label"] = normalize_labels(df["true_label"])
    df["predicted_label"] = normalize_labels(df["predicted_label"])

    model_predictions[model_name] = df.set_index("image_key")

reference = model_predictions["ViT-B/16"]

comparison = reference[
    ["source", "filename", "true_label", "visual_family_id"]
].copy()

for model_name, df in model_predictions.items():
    if set(df.index) != set(reference.index):
        raise ValueError(f"Test images differ for {model_name}")

    aligned = df.reindex(reference.index)

    if not aligned["true_label"].equals(reference["true_label"]):
        raise ValueError(f"Ground-truth mismatch: {model_name}")

    comparison[f"{model_name}_prediction"] = aligned["predicted_label"]
    comparison[f"{model_name}_correct"] = (
        aligned["predicted_label"] == comparison["true_label"]
    )

correct_cols = [
    f"{name}_correct" for name in prediction_files
]

comparison["models_correct"] = comparison[correct_cols].sum(axis=1)
comparison["models_wrong"] = 3 - comparison["models_correct"]

print("NETRASCAN.AI — CROSS-MODEL ERROR ANALYSIS")
print("-" * 65)
print("Total test images:", len(comparison))

print("\nError agreement:")
for wrong_count in range(4):
    count = (comparison["models_wrong"] == wrong_count).sum()
    print(f"{wrong_count} models wrong: {count}")

print("\nIndividual errors:")
for model_name in prediction_files:
    count = (~comparison[f"{model_name}_correct"]).sum()
    print(f"{model_name}: {count}")

print("\nUnique errors:")
for model_name in prediction_files:
    others = [m for m in prediction_files if m != model_name]

    mask = (
        (~comparison[f"{model_name}_correct"])
        & comparison[f"{others[0]}_correct"]
        & comparison[f"{others[1]}_correct"]
    )

    print(f"{model_name}: {mask.sum()}")

prediction_cols = [
    f"{name}_prediction" for name in prediction_files
]

def majority_vote(row):
    counts = row[prediction_cols].value_counts()
    return counts.index[0] if counts.iloc[0] >= 2 else "NO_MAJORITY"

comparison["ensemble_prediction"] = comparison.apply(
    majority_vote, axis=1
)

comparison["ensemble_correct"] = (
    comparison["ensemble_prediction"] == comparison["true_label"]
)

ensemble_correct = int(comparison["ensemble_correct"].sum())
ensemble_errors = len(comparison) - ensemble_correct

print("\nMAJORITY-VOTING ENSEMBLE")
print("-" * 65)
print(f"Accuracy: {ensemble_correct / len(comparison):.4%}")
print(f"Correct predictions: {ensemble_correct}")
print(f"Misclassifications: {ensemble_errors}")
print(
    "No-majority cases:",
    (comparison["ensemble_prediction"] == "NO_MAJORITY").sum()
)

comparison_dir = ROOT / "results" / "comparison_v4"
comparison_dir.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    comparison_dir / "cross_model_error_analysis.csv",
    index=False
)

print("\nPASS — Cross-model error analysis saved.")


### Plot error dashboard


In [ ]:
# STEP 56 — Plot error dashboard
# STEP 69 — Cross-Model Error Analysis Dashboard

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

models = list(prediction_files.keys())
colors = ["#F4A261", "#4ECDC4", "#5B8DEF"]

individual_errors = [
    int((~comparison[f"{m}_correct"]).sum())
    for m in models
]

unique_errors = []
for model in models:
    others = [m for m in models if m != model]
    mask = (
        (~comparison[f"{model}_correct"])
        & comparison[f"{others[0]}_correct"]
        & comparison[f"{others[1]}_correct"]
    )
    unique_errors.append(int(mask.sum()))

agreement_counts = [
    int((comparison["models_wrong"] == i).sum())
    for i in range(4)
]

accuracy_values = [
    comparison[f"{m}_correct"].mean() * 100
    for m in models
]
accuracy_values.append(comparison["ensemble_correct"].mean() * 100)

plt.style.use("dark_background")

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.patch.set_facecolor("#0B1220")

for ax in axes.flat:
    ax.set_facecolor("#151E2E")
    ax.grid(axis="y", alpha=0.15)
    ax.set_axisbelow(True)

bars = axes[0, 0].bar(
    models, individual_errors, color=colors, width=0.55
)
axes[0, 0].set_title("Misclassifications by Architecture", fontsize=15, weight="bold")
axes[0, 0].set_ylabel("Incorrect Predictions")
axes[0, 0].set_ylim(0, max(individual_errors) * 1.25)

for bar, value in zip(bars, individual_errors):
    axes[0, 0].text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.35,
        str(value),
        ha="center", fontsize=12, weight="bold"
    )

agreement_labels = [
    "All Correct", "1 Wrong", "2 Wrong", "All Wrong"
]
agreement_colors = ["#4ECDC4", "#5B8DEF", "#F4A261", "#EF476F"]

bars = axes[0, 1].bar(
    agreement_labels, agreement_counts,
    color=agreement_colors, width=0.6
)
axes[0, 1].set_title("Cross-Model Error Agreement", fontsize=15, weight="bold")
axes[0, 1].set_ylabel("Number of Test Images")

for bar, value in zip(bars, agreement_counts):
    axes[0, 1].text(
        bar.get_x() + bar.get_width() / 2,
        value + 5,
        str(value),
        ha="center", fontsize=11, weight="bold"
    )

bars = axes[1, 0].bar(
    models, unique_errors, color=colors, width=0.55
)
axes[1, 0].set_title("Unique Misclassifications", fontsize=15, weight="bold")
axes[1, 0].set_ylabel("Errors Not Shared with Other Models")
axes[1, 0].set_ylim(0, max(unique_errors) + 2)

for bar, value in zip(bars, unique_errors):
    axes[1, 0].text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.15,
        str(value),
        ha="center", fontsize=12, weight="bold"
    )

accuracy_names = models + ["Majority Vote"]
accuracy_colors = colors + ["#B388EB"]

bars = axes[1, 1].bar(
    accuracy_names, accuracy_values,
    color=accuracy_colors, width=0.6
)
axes[1, 1].set_title("Single Models vs Majority Voting", fontsize=15, weight="bold")
axes[1, 1].set_ylabel("Test Accuracy (%)")
axes[1, 1].set_ylim(0, 105)
axes[1, 1].tick_params(axis="x", labelrotation=12)

for bar, value in zip(bars, accuracy_values):
    axes[1, 1].text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}%",
        ha="center", fontsize=10, weight="bold"
    )

fig.suptitle(
    "NetraScan.AI | Cross-Model Error Analysis",
    fontsize=22, weight="bold", y=0.98
)

fig.text(
    0.5, 0.935,
    "V4 Dataset | 571 Test Images | ViT-B/16 vs Swin-Tiny vs MobileViT-S",
    ha="center", fontsize=12, color="#AAB7C4"
)

plt.tight_layout(rect=[0, 0.02, 1, 0.91])

dashboard_path = comparison_dir / "cross_model_error_dashboard.png"

plt.savefig(
    dashboard_path,
    dpi=300,
    bbox_inches="tight",
    facecolor=fig.get_facecolor()
)

plt.show()

print("PASS — Cross-model error dashboard saved.")
print("Location:", dashboard_path)


## 08 · Ensemble inference and explainability

Load the three checkpoints and configure model-specific preprocessing. Explore Swin-Tiny Grad-CAM, including stage selection and spatial activation visualization. **Heatmaps are qualitative explanations, not clinically validated lesion localization.**


### Locate saved checkpoints


In [ ]:
# STEP 57 — Locate saved checkpoints
# Cell 2: Verify trained model checkpoints

from pathlib import Path

ROOT = Path(r"C:\Personal AI & ML PROJECTS\AI IN HEALTHCARE\MY PROJECT E PAARVAI")

CHECKPOINTS = {
    "ViT-B/16": ROOT / "models" / "vit_v4" / "vit_v4_best.pth",
    "Swin-Tiny": ROOT / "models" / "swin_v4" / "swin_v4_best.pth",
    "MobileViT-S": ROOT / "models" / "mobilevit_v4" / "mobilevit_v4_best.pth"
}

print("NETRASCAN.AI — MODEL CHECKPOINT VERIFICATION")
print("-" * 55)

for name, path in CHECKPOINTS.items():
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"PASS | {name:12} | {size_mb:.2f} MB")
    else:
        print(f"MISSING | {name:12} | {path}")

assert all(path.is_file() for path in CHECKPOINTS.values()), "Missing model checkpoint."

print("\nPASS — All three trained models are available.")


### Load inference models


In [ ]:
# STEP 58 — Load inference models
# Cell 3: Load all three trained models for inference

import torch
import timm
import gc

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_ARCHITECTURES = {
    "ViT-B/16": "vit_base_patch16_224",
    "Swin-Tiny": "swin_tiny_patch4_window7_224",
    "MobileViT-S": "mobilevit_s"
}

CLASS_NAMES = ["normal", "immature", "mature"]

inference_models = {}

print("NETRASCAN.AI — LOADING TRAINED MODELS")
print("-" * 55)
print("Device:", DEVICE)

for name, architecture in MODEL_ARCHITECTURES.items():

    model = timm.create_model(
        architecture,
        pretrained=False,
        num_classes=3
    )

    checkpoint = torch.load(
        CHECKPOINTS[name],
        map_location="cpu",
        weights_only=True
    )

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    model.load_state_dict(state_dict, strict=True)

    model = model.to(DEVICE)
    model.eval()

    inference_models[name] = model

    parameters = sum(p.numel() for p in model.parameters())

    print(f"PASS | {name}")
    print(f"     Architecture: {architecture}")
    print(f"     Parameters: {parameters:,}")
    print(f"     Inference mode: {not model.training}")

    del checkpoint, state_dict
    gc.collect()

if DEVICE.type == "cuda":
    print(
        "\nGPU memory allocated:",
        f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
    )

print("\nPASS — All three models loaded successfully.")


### Configure inference preprocessing


In [ ]:
# STEP 59 — Configure inference preprocessing
# Cell 4: Configure model-specific image preprocessing

from torchvision import transforms
from PIL import Image
import torch

inference_transforms = {
    "ViT-B/16": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.5, 0.5, 0.5),
            std=(0.5, 0.5, 0.5)
        )
    ]),

    "Swin-Tiny": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        )
    ]),

    "MobileViT-S": transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.0, 0.0, 0.0),
            std=(1.0, 1.0, 1.0)
        )
    ])
}

test_image = Image.new("RGB", (640, 480), color=(128, 128, 128))

print("NETRASCAN.AI — INFERENCE PREPROCESSING")
print("-" * 55)

for name, transform in inference_transforms.items():
    tensor = transform(test_image).unsqueeze(0)

    expected_size = (
        256 if name == "MobileViT-S" else 224
    )

    assert tensor.shape == (
        1, 3, expected_size, expected_size
    )

    print(f"PASS | {name:12} | Shape: {tuple(tensor.shape)}")
    print(f"       Range: [{tensor.min():.3f}, {tensor.max():.3f}]")

print("\nPASS — All inference transformations verified.")


### Implement ensemble inference


In [ ]:
# STEP 60 — Implement ensemble inference
# Cell 5: Build the three-model inference engine

import time
import numpy as np
import pandas as pd
from PIL import Image

CLASS_NAMES = ["normal", "immature", "mature"]

@torch.inference_mode()
def predict_cataract(image):
    if image is None:
        raise ValueError("Please upload an eye image.")

    if isinstance(image, (str, Path)):
        image = Image.open(image)

    image = image.convert("RGB")

    predictions = {}
    probability_rows = []
    timing_rows = []

    for name, model in inference_models.items():
        input_tensor = (
            inference_transforms[name](image)
            .unsqueeze(0)
            .to(DEVICE)
        )

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        logits = model(input_tensor)
        probabilities = torch.softmax(logits, dim=1)[0]

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        elapsed_ms = (time.perf_counter() - start_time) * 1000

        predicted_index = int(torch.argmax(probabilities).item())
        predicted_class = CLASS_NAMES[predicted_index]
        confidence = float(probabilities[predicted_index].item())

        predictions[name] = {
            "predicted_class": predicted_class,
            "confidence": confidence,
            "probabilities": {
                class_name: float(probabilities[i].item())
                for i, class_name in enumerate(CLASS_NAMES)
            },
            "inference_ms": elapsed_ms
        }

        probability_rows.append({
            "Model": name,
            **{
                class_name.capitalize(): round(
                    predictions[name]["probabilities"][class_name] * 100, 2
                )
                for class_name in CLASS_NAMES
            }
        })

        timing_rows.append({
            "Model": name,
            "Prediction": predicted_class.capitalize(),
            "Confidence (%)": round(confidence * 100, 2),
            "Inference (ms)": round(elapsed_ms, 2)
        })

    votes = [
        result["predicted_class"]
        for result in predictions.values()
    ]

    vote_counts = {
        class_name: votes.count(class_name)
        for class_name in CLASS_NAMES
    }

    majority_class = max(vote_counts, key=vote_counts.get)
    majority_count = vote_counts[majority_class]

    ensemble_prediction = (
        majority_class if majority_count >= 2 else "No majority"
    )

    return {
        "predictions": predictions,
        "ensemble_prediction": ensemble_prediction,
        "agreement": f"{majority_count}/3" if majority_count >= 2 else "1/3",
        "probability_table": pd.DataFrame(probability_rows),
        "summary_table": pd.DataFrame(timing_rows)
    }

print("NETRASCAN.AI — INFERENCE ENGINE")
print("-" * 55)

test_image = Image.new("RGB", (640, 480), color=(128, 128, 128))
test_result = predict_cataract(test_image)

print(test_result["summary_table"].to_string(index=False))
print("\nMajority vote:", test_result["ensemble_prediction"])
print("Agreement:", test_result["agreement"])
print("\nPASS — Three-model inference engine operational.")


### Inspect Swin feature stages


In [ ]:
# STEP 61 — Inspect Swin feature stages
# Cell 9: Inspect Swin-Tiny architecture and select Grad-CAM target layer

import torch

print("NETRASCAN.AI — SWIN-TINY XAI LAYER INSPECTION")
print("-" * 65)

swin_model = inference_models["Swin-Tiny"]
swin_model.eval()

print("Model architecture:", type(swin_model).__name__)
print("Device:", next(swin_model.parameters()).device)

print("\nSWIN TRANSFORMER STAGES")
print("-" * 65)

for stage_index, stage in enumerate(swin_model.layers):
    print(
        f"Stage {stage_index}: "
        f"{len(stage.blocks)} transformer blocks"
    )

print("\nFINAL TRANSFORMER STAGE")
print("-" * 65)

final_stage = swin_model.layers[-1]
final_block = final_stage.blocks[-1]

print("Final block:", type(final_block).__name__)
print("Final block normalization:", final_block.norm1)

target_layers = [final_block.norm1]

print("\nSELECTED GRAD-CAM TARGET")
print("-" * 65)
print("Layer: swin_model.layers[-1].blocks[-1].norm1")
print("Layer type:", type(target_layers[0]).__name__)

captured_shapes = []

def inspect_swin_features(module, inputs, output):
    captured_shapes.append(tuple(output.shape))

hook = target_layers[0].register_forward_hook(
    inspect_swin_features
)

try:
    dummy_input = torch.randn(
        1, 3, 224, 224,
        device=DEVICE
    )

    with torch.no_grad():
        dummy_output = swin_model(dummy_input)

finally:
    hook.remove()

print("\nFEATURE SHAPE VERIFICATION")
print("-" * 65)

print("Input shape:", tuple(dummy_input.shape))
print("Target layer output shapes:", captured_shapes)
print("Classification output:", tuple(dummy_output.shape))

assert len(captured_shapes) == 1, (
    "Expected exactly one target-layer activation."
)

assert dummy_output.shape == (1, 3), (
    "Expected three cataract class logits."
)

print("\nPASS — Swin-Tiny target layer verified.")
print("PASS — Ready to configure Grad-CAM reshape transform.")


### Configure Grad-CAM


In [ ]:
# STEP 62 — Configure Grad-CAM
# Cell 10: Configure and verify Swin-Tiny Grad-CAM reshape transform

import torch
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

print("NETRASCAN.AI — GRAD-CAM CONFIGURATION")
print("-" * 65)

def swin_reshape_transform(tensor):
    if tensor.ndim != 4:
        raise ValueError(
            f"Expected 4D Swin activations, got {tuple(tensor.shape)}"
        )

    if tensor.shape[-1] != 768:
        raise ValueError(
            f"Expected 768 feature channels, got {tuple(tensor.shape)}"
        )

    return tensor.permute(0, 3, 1, 2).contiguous()


test_activation = torch.randn(
    1, 7, 7, 768,
    device=DEVICE
)

reshaped_activation = swin_reshape_transform(test_activation)

print("Original shape:", tuple(test_activation.shape))
print("Reshaped shape:", tuple(reshaped_activation.shape))

assert reshaped_activation.shape == (1, 768, 7, 7)

print("\nINITIALIZING GRAD-CAM")
print("-" * 65)

swin_cam = GradCAM(
    model=swin_model,
    target_layers=target_layers,
    reshape_transform=swin_reshape_transform
)

print("Target layer:", target_layers[0])
print("Grad-CAM object:", type(swin_cam).__name__)

print("\nTESTING GRADIENT-BASED EXPLANATION")
print("-" * 65)

test_input = torch.randn(
    1, 3, 224, 224,
    device=DEVICE
)

with torch.no_grad():
    test_logits = swin_model(test_input)
    predicted_index = int(test_logits.argmax(dim=1).item())

print("Predicted class index:", predicted_index)

test_targets = [
    ClassifierOutputTarget(predicted_index)
]

test_cam = swin_cam(
    input_tensor=test_input,
    targets=test_targets
)

print("Grad-CAM output shape:", test_cam.shape)
print("Minimum activation:", float(test_cam.min()))
print("Maximum activation:", float(test_cam.max()))

assert test_cam.shape == (1, 224, 224)
assert torch.isfinite(torch.as_tensor(test_cam)).all()

print("\nPASS — Swin-Tiny Grad-CAM initialized.")
print("PASS — NHWC to NCHW transformation verified.")
print("PASS — Gradient-based heatmap generated successfully.")


### Generate Grad-CAM


In [ ]:
# STEP 63 — Generate Grad-CAM
# Cell 11: Generate and visualize Swin-Tiny Grad-CAM heatmap

import numpy as np
import matplotlib.pyplot as plt
import torch

from PIL import Image
from pathlib import Path
from tkinter import Tk, filedialog

from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

print("NETRASCAN.AI — SWIN-TINY GRAD-CAM VISUALIZATION")
print("-" * 65)

root_window = Tk()
root_window.withdraw()
root_window.attributes("-topmost", True)

image_path = filedialog.askopenfilename(
    title="Select an Eye Image for NetraScan.AI XAI",
    filetypes=[
        ("Image files", "*.jpg *.jpeg *.png *.bmp *.webp"),
        ("All files", "*.*")
    ]
)

root_window.destroy()

if not image_path:
    raise RuntimeError("No image selected. Run Cell 11 again.")

print("Selected image:", image_path)

original_image = Image.open(image_path).convert("RGB")
display_image = original_image.resize((224, 224))

input_tensor = inference_transforms["Swin-Tiny"](
    original_image
).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    logits = swin_model(input_tensor)
    probabilities = torch.softmax(logits, dim=1)[0]

predicted_index = int(probabilities.argmax().item())
predicted_class = CLASS_NAMES[predicted_index]
confidence = float(probabilities[predicted_index].item()) * 100

print("\nSWIN-TINY PREDICTION")
print("-" * 65)
print("Predicted class:", predicted_class)
print(f"Confidence: {confidence:.2f}%")

cam_targets = [
    ClassifierOutputTarget(predicted_index)
]

grayscale_cam = swin_cam(
    input_tensor=input_tensor,
    targets=cam_targets
)[0]

rgb_image = np.asarray(display_image, dtype=np.float32) / 255.0

heatmap_overlay = show_cam_on_image(
    rgb_image,
    grayscale_cam,
    use_rgb=True,
    image_weight=0.55
)

plt.figure(figsize=(16, 5))

plt.subplot(1, 3, 1)
plt.imshow(display_image)
plt.title("Original Eye Image", fontsize=13)
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(grayscale_cam, cmap="jet", vmin=0, vmax=1)
plt.title("Grad-CAM Activation Heatmap", fontsize=13)
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(heatmap_overlay)
plt.title(
    f"Swin-Tiny: {predicted_class}\n"
    f"Confidence: {confidence:.2f}%",
    fontsize=13
)
plt.axis("off")

plt.suptitle(
    "NetraScan.AI — Explainable AI (Swin-Tiny)",
    fontsize=17,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

print("\nPASS — Grad-CAM heatmap generated.")
print("PASS — Original, activation, and overlay visualizations displayed.")


### Save Grad-CAM overlays


In [ ]:
# STEP 64 — Save Grad-CAM overlays
# Cell 12: Save and display Swin-Tiny Grad-CAM visualization

from pathlib import Path
from IPython.display import display, Image as IPythonImage
import matplotlib.pyplot as plt

print("NETRASCAN.AI — SAVE AND DISPLAY XAI RESULTS")
print("-" * 60)

output_dir = Path("results/xai")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "swin_gradcam_test.png"

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(display_image)
axes[0].set_title("Original Eye Image")

axes[1].imshow(
    grayscale_cam,
    cmap="jet",
    vmin=0,
    vmax=1
)
axes[1].set_title("Grad-CAM Heatmap")

axes[2].imshow(heatmap_overlay)
axes[2].set_title(
    f"Swin-Tiny: {predicted_class}\n"
    f"Confidence: {confidence:.2f}%"
)

for ax in axes:
    ax.axis("off")

fig.suptitle(
    "NetraScan.AI — Swin-Tiny Explainable AI",
    fontsize=17,
    fontweight="bold"
)

fig.tight_layout()

fig.savefig(
    output_path,
    dpi=180,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(fig)

print("Saved:", output_path.resolve())
print("File size:", round(output_path.stat().st_size / 1024, 2), "KB")

display(IPythonImage(filename=str(output_path)))

print("\nPASS — XAI visualization saved and displayed.")


### Compare Grad-CAM stages


In [ ]:
# STEP 65 — Compare Grad-CAM stages
# Cell 13: Compare Swin-Tiny Stage 2 and Stage 3 Grad-CAM

import numpy as np
import matplotlib.pyplot as plt
import torch

from pathlib import Path
from IPython.display import display, Image as IPythonImage
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

print("NETRASCAN.AI — SWIN-TINY XAI STAGE COMPARISON")
print("-" * 65)

swin_model = inference_models["Swin-Tiny"]
swin_model.eval()

stage2_layer = swin_model.layers[2].blocks[-1].norm1
stage3_layer = swin_model.layers[3].blocks[-1].norm1

def swin_cam_reshape(tensor):
    if tensor.ndim != 4:
        raise ValueError(
            f"Expected NHWC feature tensor, got {tuple(tensor.shape)}"
        )
    return tensor.permute(0, 3, 1, 2).contiguous()

input_tensor = inference_transforms["Swin-Tiny"](
    original_image
).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    logits = swin_model(input_tensor)
    probs = torch.softmax(logits, dim=1)[0]
    predicted_index = int(probs.argmax().item())
    predicted_class = CLASS_NAMES[predicted_index]
    confidence = float(probs[predicted_index].item()) * 100

targets = [ClassifierOutputTarget(predicted_index)]

print("Image:", Path(image_path).name)
print("Prediction:", predicted_class)
print(f"Confidence: {confidence:.2f}%")

stage_results = {}

for stage_name, layer in [
    ("Stage 2", stage2_layer),
    ("Stage 3", stage3_layer)
]:
    print(f"\nGenerating {stage_name} Grad-CAM...")

    cam = GradCAM(
        model=swin_model,
        target_layers=[layer],
        reshape_transform=swin_cam_reshape
    )

    heatmap = cam(
        input_tensor=input_tensor,
        targets=targets
    )[0]

    stage_results[stage_name] = heatmap

    print(
        f"PASS — {stage_name}: "
        f"heatmap shape {heatmap.shape}, "
        f"range [{heatmap.min():.3f}, {heatmap.max():.3f}]"
    )

rgb_image = np.asarray(
    original_image.resize((224, 224)),
    dtype=np.float32
) / 255.0

stage2_overlay = show_cam_on_image(
    rgb_image,
    stage_results["Stage 2"],
    use_rgb=True,
    image_weight=0.55
)

stage3_overlay = show_cam_on_image(
    rgb_image,
    stage_results["Stage 3"],
    use_rgb=True,
    image_weight=0.55
)

fig, axes = plt.subplots(1, 5, figsize=(20, 4.8))

images = [
    rgb_image,
    stage_results["Stage 2"],
    stage2_overlay,
    stage_results["Stage 3"],
    stage3_overlay
]

titles = [
    "Original Eye",
    "Stage 2 Heatmap",
    "Stage 2 Overlay",
    "Stage 3 Heatmap",
    "Stage 3 Overlay"
]

for index, (ax, image, title) in enumerate(
    zip(axes, images, titles)
):
    if index in [1, 3]:
        ax.imshow(image, cmap="jet", vmin=0, vmax=1)
    else:
        ax.imshow(image)

    ax.set_title(title, fontsize=12)
    ax.axis("off")

fig.suptitle(
    f"NetraScan.AI — Swin-Tiny Grad-CAM Comparison\n"
    f"Prediction: {predicted_class} | Confidence: {confidence:.2f}%",
    fontsize=15,
    fontweight="bold"
)

fig.tight_layout()

output_dir = Path("results/xai")
output_dir.mkdir(parents=True, exist_ok=True)

comparison_path = output_dir / "swin_stage2_vs_stage3_gradcam.png"

fig.savefig(
    comparison_path,
    dpi=180,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(fig)

display(IPythonImage(filename=str(comparison_path)))

print("\nSaved comparison:", comparison_path.resolve())
print("PASS — Stage 2 and Stage 3 XAI comparison completed.")


## 09 · Final standalone prototype implementation

These source-generating cells document the portable inference engine, Grad-CAM module, deterministic report generator, and Gradio application. They write files to a Kingston SSD path defined in the original development environment. **Review and change that destination before running.** Deployment installation, repair, and server-launch diagnostics have been removed.


### Export portable inference module


In [ ]:
# STEP 66 — Export portable inference module
# Cell 23: Create standalone NetraScan.AI inference engine on Kingston SSD

from pathlib import Path
import ast

print("NETRASCAN.AI — STANDALONE INFERENCE ENGINE")
print("=" * 65)

SSD_ROOT = Path("E:/NetraScan.AI_PORTABLE")
INFERENCE_FILE = SSD_ROOT / "inference.py"

assert (SSD_ROOT / "models/vit_v4_best.pth").is_file()
assert (SSD_ROOT / "models/swin_v4_best.pth").is_file()
assert (SSD_ROOT / "models/mobilevit_v4_best.pth").is_file()

inference_source = r'''
from pathlib import Path
from collections import Counter
from time import perf_counter

import torch
import timm
from PIL import Image
from torchvision import transforms


APP_ROOT = Path(__file__).resolve().parent

CLASS_NAMES = ["normal", "immature", "mature"]

MODEL_CONFIGS = {
    "ViT-B/16": {
        "architecture": "vit_base_patch16_224",
        "checkpoint": "vit_v4_best.pth",
        "size": 224,
        "mean": (0.5, 0.5, 0.5),
        "std": (0.5, 0.5, 0.5),
    },
    "Swin-Tiny": {
        "architecture": "swin_tiny_patch4_window7_224",
        "checkpoint": "swin_v4_best.pth",
        "size": 224,
        "mean": (0.485, 0.456, 0.406),
        "std": (0.229, 0.224, 0.225),
    },
    "MobileViT-S": {
        "architecture": "mobilevit_s",
        "checkpoint": "mobilevit_v4_best.pth",
        "size": 256,
        "mean": (0.0, 0.0, 0.0),
        "std": (1.0, 1.0, 1.0),
    },
}


class NetraScanInference:

    def __init__(self, device=None):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"

        self.device = torch.device(device)
        self.models = {}
        self.transforms = {}

        print(f"NetraScan.AI device: {self.device}")

        if self.device.type == "cuda":
            print("GPU:", torch.cuda.get_device_name(0))

        self.load_models()

    def load_models(self):

        for name, config in MODEL_CONFIGS.items():

            print(f"Loading {name}...")

            model = timm.create_model(
                config["architecture"],
                pretrained=False,
                num_classes=len(CLASS_NAMES),
            )

            checkpoint_path = (
                APP_ROOT / "models" / config["checkpoint"]
            )

            if not checkpoint_path.is_file():
                raise FileNotFoundError(checkpoint_path)

            checkpoint = torch.load(
                checkpoint_path,
                map_location="cpu",
                weights_only=True,
            )

            if isinstance(checkpoint, dict):
                if "model_state_dict" in checkpoint:
                    checkpoint = checkpoint["model_state_dict"]
                elif "state_dict" in checkpoint:
                    checkpoint = checkpoint["state_dict"]

            model.load_state_dict(checkpoint, strict=True)

            model.to(self.device)
            model.eval()

            self.models[name] = model

            self.transforms[name] = transforms.Compose([
                transforms.Resize(
                    (config["size"], config["size"])
                ),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=config["mean"],
                    std=config["std"],
                ),
            ])

            print(f"PASS — {name} loaded")

    @torch.inference_mode()
    def predict(self, image):

        if isinstance(image, (str, Path)):
            with Image.open(image) as source:
                image = source.convert("RGB")
        elif isinstance(image, Image.Image):
            image = image.convert("RGB")
        else:
            raise TypeError(
                "Expected a PIL image or image file path."
            )

        predictions = {}
        votes = []

        for name, model in self.models.items():

            tensor = self.transforms[name](image)
            tensor = tensor.unsqueeze(0).to(self.device)

            if self.device.type == "cuda":
                torch.cuda.synchronize()

            start = perf_counter()

            logits = model(tensor)
            probabilities = torch.softmax(logits, dim=1)[0]

            if self.device.type == "cuda":
                torch.cuda.synchronize()

            elapsed_ms = (perf_counter() - start) * 1000

            index = int(probabilities.argmax().item())
            predicted_class = CLASS_NAMES[index]

            confidence = float(probabilities[index].item())

            predictions[name] = {
                "predicted_class": predicted_class,
                "confidence": confidence,
                "probabilities": {
                    class_name: float(probabilities[i].item())
                    for i, class_name in enumerate(CLASS_NAMES)
                },
                "inference_ms": round(elapsed_ms, 2),
            }

            votes.append(predicted_class)

        counts = Counter(votes)
        ensemble_prediction, vote_count = counts.most_common(1)[0]

        if vote_count < 2:
            ensemble_prediction = "inconclusive"

        return {
            "ensemble_prediction": ensemble_prediction,
            "agreement": f"{vote_count}/3",
            "predictions": predictions,
        }
'''

print("\n1. VALIDATING PYTHON SOURCE")
print("-" * 65)

ast.parse(inference_source)
print("PASS — Python syntax validated.")

print("\n2. WRITING INFERENCE MODULE")
print("-" * 65)

INFERENCE_FILE.write_text(
    inference_source.lstrip(),
    encoding="utf-8"
)

print("Created:", INFERENCE_FILE)
print(
    "Size:",
    round(INFERENCE_FILE.stat().st_size / 1024, 2),
    "KB"
)

print("\n3. VERIFYING MODEL CONFIGURATIONS")
print("-" * 65)

assert "vit_base_patch16_224" in inference_source
assert "swin_tiny_patch4_window7_224" in inference_source
assert "mobilevit_s" in inference_source
assert "torch.cuda.is_available()" in inference_source

print("PASS — ViT-B/16 configured.")
print("PASS — Swin-Tiny configured.")
print("PASS — MobileViT-S configured.")
print("PASS — CUDA/CPU fallback configured.")
print("PASS — Relative checkpoint paths configured.")

print("\nPASS — Standalone inference module created successfully.")


### Export portable explainability module


In [ ]:
# STEP 67 — Export portable explainability module
# Cell 25: Create standalone Swin-Tiny Grad-CAM module on Kingston SSD

from pathlib import Path
import ast

print("NETRASCAN.AI — PORTABLE EXPLAINABLE AI MODULE")
print("=" * 65)

SSD_ROOT = Path("E:/NetraScan.AI_PORTABLE")
XAI_FILE = SSD_ROOT / "explainability.py"

assert (SSD_ROOT / "inference.py").is_file()

xai_source = r'''
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


def swin_stage2_reshape(tensor):
    """
    Convert Swin Stage 2 NHWC features into NCHW format.
    Expected output: [B, C, H, W].
    """
    if tensor.ndim == 4:
        return tensor.permute(0, 3, 1, 2).contiguous()

    if tensor.ndim == 3:
        batch, tokens, channels = tensor.shape
        side = int(tokens ** 0.5)

        if side * side != tokens:
            raise ValueError(
                f"Cannot reshape {tokens} tokens into a square grid."
            )

        return tensor.transpose(1, 2).reshape(
            batch, channels, side, side
        )

    raise ValueError(
        f"Unexpected Swin feature shape: {tuple(tensor.shape)}"
    )


class NetraScanXAI:

    def __init__(self, inference_engine):
        self.engine = inference_engine

        self.model = inference_engine.models["Swin-Tiny"]
        self.device = inference_engine.device

        self.target_layer = (
            self.model.layers[2].blocks[-1].norm1
        )

        print("PASS — Swin-Tiny Stage 2 Grad-CAM target configured.")

    def generate(self, image, class_index=None):

        if isinstance(image, (str, Path)):
            with Image.open(image) as source:
                image = source.convert("RGB")
        elif isinstance(image, Image.Image):
            image = image.convert("RGB")
        else:
            raise TypeError(
                "Expected PIL image or image file path."
            )

        image_size = 224

        resized_image = image.resize(
            (image_size, image_size),
            Image.Resampling.BILINEAR
        )

        rgb_float = (
            np.asarray(resized_image, dtype=np.float32) / 255.0
        )

        input_tensor = self.engine.transforms["Swin-Tiny"](
            image
        ).unsqueeze(0).to(self.device)

        if class_index is None:
            with torch.no_grad():
                logits = self.model(input_tensor)
                class_index = int(logits.argmax(dim=1).item())

        from pytorch_grad_cam.utils.model_targets import (
            ClassifierOutputTarget
        )

        targets = [ClassifierOutputTarget(class_index)]

        with torch.enable_grad():
            with GradCAM(
                model=self.model,
                target_layers=[self.target_layer],
                reshape_transform=swin_stage2_reshape
            ) as cam:

                grayscale_cam = cam(
                    input_tensor=input_tensor,
                    targets=targets
                )[0]

        overlay = show_cam_on_image(
            rgb_float,
            grayscale_cam,
            use_rgb=True,
            image_weight=0.55
        )

        import cv2

        heatmap_bgr = cv2.applyColorMap(
            np.uint8(255 * grayscale_cam),
            cv2.COLORMAP_JET
        )

        heatmap_rgb = cv2.cvtColor(
            heatmap_bgr,
            cv2.COLOR_BGR2RGB
        )

        return {
            "overlay": Image.fromarray(overlay),
            "heatmap": Image.fromarray(heatmap_rgb),
            "class_index": class_index,
            "cam": grayscale_cam
        }
'''

print("\n1. VALIDATING MODULE")
print("-" * 65)

ast.parse(xai_source)
print("PASS — Python syntax valid.")

print("\n2. WRITING FILE")
print("-" * 65)

XAI_FILE.write_text(
    xai_source.lstrip(),
    encoding="utf-8"
)

print("Created:", XAI_FILE)
print("Size:", round(XAI_FILE.stat().st_size / 1024, 2), "KB")

print("\n3. VERIFYING CONFIGURATION")
print("-" * 65)

assert "self.model.layers[2].blocks[-1].norm1" in xai_source
assert "torch.enable_grad()" in xai_source
assert "swin_stage2_reshape" in xai_source

print("PASS — Stage 2 target selected.")
print("PASS — Gradient-based explanation enabled.")
print("PASS — NHWC-to-NCHW reshape implemented.")
print("PASS — Heatmap and overlay outputs configured.")

print("\nPASS — Portable XAI module created.")


### Export deterministic report generator


In [ ]:
# STEP 68 — Export deterministic report generator
# Cell 27: Create standalone NetraScan.AI report generator on Kingston SSD

from pathlib import Path
import ast

print("NETRASCAN.AI — PORTABLE REPORT GENERATOR")
print("=" * 65)

SSD_ROOT = Path(r"E:\NetraScan.AI_PORTABLE")
REPORT_FILE = SSD_ROOT / "report_generator.py"

assert SSD_ROOT.is_dir()

report_source = r'''
from datetime import datetime
from pathlib import Path
import re


APP_ROOT = Path(__file__).resolve().parent

CLASS_LABELS = {
    "normal": "Normal",
    "immature": "Immature Cataract",
    "mature": "Mature Cataract",
    "inconclusive": "Inconclusive"
}


def generate_report(result, image_name="Uploaded image", save=True):
    """
    Generate a research-oriented Markdown inference report.

    Parameters
    ----------
    result : dict
        Output from NetraScanInference.predict().
    image_name : str
        Original image filename or identifier.
    save : bool
        Whether to save the report to the portable reports folder.

    Returns
    -------
    dict
        Report text and optional saved file path.
    """

    timestamp = datetime.now().astimezone()

    ensemble = result["ensemble_prediction"]
    agreement = result["agreement"]
    predictions = result["predictions"]

    ensemble_label = CLASS_LABELS.get(
        ensemble, str(ensemble).title()
    )

    lines = [
        "# NetraScan.AI — Cataract Classification Report",
        "",
        f"**Generated:** {timestamp.strftime('%d %B %Y, %I:%M:%S %p %Z')}",
        f"**Input image:** {Path(str(image_name)).name}",
        "**System:** Multi-Transformer Cataract Classification",
        "",
        "---",
        "",
        "## 1. Ensemble Classification",
        "",
        f"**Predicted category:** {ensemble_label}",
        f"**Model agreement:** {agreement}",
        "",
        "The ensemble decision is determined by majority voting "
        "across three independently trained transformer architectures.",
        "",
        "## 2. Individual Model Predictions",
        "",
        "| Architecture | Prediction | Confidence | Inference Time |",
        "|---|---|---:|---:|"
    ]

    for model_name, details in predictions.items():

        prediction = CLASS_LABELS.get(
            details["predicted_class"],
            str(details["predicted_class"]).title()
        )

        confidence = details["confidence"] * 100
        latency = details["inference_ms"]

        lines.append(
            f"| {model_name} | {prediction} | "
            f"{confidence:.2f}% | {latency:.2f} ms |"
        )

    lines.extend([
        "",
        "## 3. Model Architecture Summary",
        "",
        "- **ViT-B/16:** Vision Transformer using global "
        "self-attention over image patches.",
        "- **Swin-Tiny:** Hierarchical Vision Transformer "
        "using shifted-window attention.",
        "- **MobileViT-S:** Lightweight hybrid architecture "
        "combining convolutional operations with transformer blocks.",
        "",
        "## 4. Explainable AI — Grad-CAM",
        "",
        "The system supports Grad-CAM visualization using "
        "the Swin-Tiny Stage 2 feature representation.",
        "",
        "Warmer heatmap regions indicate image locations "
        "associated with stronger class-related model activations. "
        "These visualizations are not validated clinical "
        "localizations and may include irrelevant image regions.",
        "",
        "## 5. Interpretation",
        ""
    ])

    vote_count = int(str(agreement).split("/")[0])

    if ensemble == "inconclusive" or vote_count < 2:

        lines.append(
            "The three models did not establish a majority "
            "classification. The output is inconclusive."
        )

    elif vote_count == 3:

        lines.append(
            f"All three architectures predicted "
            f"**{ensemble_label}**. This represents unanimous "
            "model agreement for the supplied image, but does "
            "not establish clinical certainty."
        )

    else:

        lines.append(
            f"Two of the three architectures predicted "
            f"**{ensemble_label}**. One model disagreed, "
            "indicating inter-model prediction variability."
        )

    lines.extend([
        "",
        "## 6. Research Limitations",
        "",
        "- Performance on new cameras, lighting conditions, "
        "populations, and acquisition devices has not been "
        "established by this individual inference.",
        "- Model confidence is not equivalent to a calibrated "
        "probability of disease.",
        "- Grad-CAM does not independently verify clinical correctness.",
        "- The system cannot replace ophthalmic examination.",
        "",
        "## 7. Disclaimer",
        "",
        "**NetraScan.AI is an academic research prototype.** "
        "Its predictions and visual explanations are intended "
        "for demonstration and research only. They must not "
        "be used for clinical diagnosis, treatment decisions, "
        "or medical triage.",
        "",
        "---",
        "",
        "*Generated by the NetraScan.AI offline reporting engine.*"
    ])

    report_text = "\n".join(lines)

    saved_path = None

    if save:

        output_dir = APP_ROOT / "reports"
        output_dir.mkdir(parents=True, exist_ok=True)

        safe_name = re.sub(
            r"[^A-Za-z0-9_-]+",
            "_",
            Path(str(image_name)).stem
        ).strip("_")[:50] or "image"

        filename = (
            f"NetraScan_{safe_name}_"
            f"{timestamp.strftime('%Y%m%d_%H%M%S_%f')}.md"
        )

        saved_path = output_dir / filename

        saved_path.write_text(
            report_text,
            encoding="utf-8"
        )

    return {
        "text": report_text,
        "path": str(saved_path) if saved_path else None
    }
'''

print("\n1. VALIDATING PYTHON SOURCE")
print("-" * 65)

ast.parse(report_source)
print("PASS — Python syntax validated.")

print("\n2. WRITING REPORT MODULE")
print("-" * 65)

REPORT_FILE.write_text(
    report_source.lstrip(),
    encoding="utf-8"
)

print("Created:", REPORT_FILE)
print("Size:", round(REPORT_FILE.stat().st_size / 1024, 2), "KB")

print("\n3. VERIFYING REPORT FEATURES")
print("-" * 65)

features = {
    "Ensemble classification": "Ensemble Classification",
    "Individual model results": "Individual Model Predictions",
    "Architecture explanations": "Model Architecture Summary",
    "Grad-CAM interpretation": "Explainable AI",
    "Research limitations": "Research Limitations",
    "Medical disclaimer": "academic research prototype",
    "Offline report saving": "write_text"
}

for feature, keyword in features.items():
    assert keyword in report_source
    print(f"PASS — {feature}")

print("\nPASS — Standalone report generator created successfully.")


### Export standalone Gradio application


In [ ]:
# STEP 69 — Export standalone Gradio application
# Cell 29: Create standalone NetraScan.AI Gradio application on Kingston SSD

from pathlib import Path
import ast

print("NETRASCAN.AI — STANDALONE GRADIO APPLICATION")
print("=" * 65)

SSD_ROOT = Path(r"E:\NetraScan.AI_PORTABLE")
APP_FILE = SSD_ROOT / "app.py"

required_files = [
    "inference.py",
    "explainability.py",
    "report_generator.py"
]

for filename in required_files:
    assert (SSD_ROOT / filename).is_file(), (
        f"Missing required module: {filename}"
    )

app_source = r'''
from pathlib import Path
import os

import gradio as gr
import pandas as pd
import plotly.graph_objects as go

from inference import NetraScanInference
from explainability import NetraScanXAI
from report_generator import generate_report


APP_ROOT = Path(__file__).resolve().parent
REPORTS_DIR = APP_ROOT / "reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_LABELS = {
    "normal": "Normal",
    "immature": "Immature Cataract",
    "mature": "Mature Cataract",
    "inconclusive": "Inconclusive"
}

MODEL_NAMES = [
    "ViT-B/16",
    "Swin-Tiny",
    "MobileViT-S"
]

engine = None
xai_engine = None


def initialize():
    global engine, xai_engine

    if engine is None:
        engine = NetraScanInference()
        xai_engine = NetraScanXAI(engine)

    return engine, xai_engine


def create_probability_chart(predictions):

    fig = go.Figure()

    class_names = [
        "Normal",
        "Immature",
        "Mature"
    ]

    class_keys = [
        "normal",
        "immature",
        "mature"
    ]

    for model_name in MODEL_NAMES:

        probabilities = predictions[
            model_name
        ]["probabilities"]

        values = [
            probabilities[key] * 100
            for key in class_keys
        ]

        fig.add_trace(
            go.Bar(
                name=model_name,
                x=class_names,
                y=values,
                text=[f"{v:.1f}%" for v in values],
                textposition="auto"
            )
        )

    fig.update_layout(
        template="plotly_dark",
        barmode="group",
        title="Class Probability Comparison",
        yaxis_title="Probability (%)",
        yaxis_range=[0, 105],
        xaxis_title="Predicted Class",
        paper_bgcolor="#101625",
        plot_bgcolor="#101625",
        font=dict(color="#E6E9F2"),
        legend=dict(orientation="h", y=1.15),
        margin=dict(l=35, r=20, t=75, b=40),
        height=380
    )

    return fig


def run_analysis(image):

    if image is None:
        raise gr.Error(
            "Please upload or capture an eye image first."
        )

    model_engine, explanation_engine = initialize()

    result = model_engine.predict(image)

    predictions = result["predictions"]

    ensemble_key = result["ensemble_prediction"]

    ensemble_label = CLASS_LABELS.get(
        ensemble_key,
        ensemble_key.title()
    )

    summary = (
        f"## Ensemble Prediction: {ensemble_label}\n\n"
        f"**Model agreement:** {result['agreement']}\n\n"
        "Academic research prototype — not a clinical diagnosis."
    )

    rows = []

    for model_name in MODEL_NAMES:

        details = predictions[model_name]

        rows.append({
            "Model": model_name,
            "Prediction": CLASS_LABELS.get(
                details["predicted_class"],
                details["predicted_class"]
            ),
            "Confidence (%)": round(
                details["confidence"] * 100, 2
            ),
            "Inference (ms)": details["inference_ms"]
        })

    dataframe = pd.DataFrame(rows)

    chart = create_probability_chart(predictions)

    overlay = None
    heatmap = None
    xai_status = "Grad-CAM not generated."

    try:

        explanation = explanation_engine.generate(image)

        overlay = explanation["overlay"]
        heatmap = explanation["heatmap"]

        xai_status = (
            "Swin-Tiny Stage 2 Grad-CAM generated successfully."
        )

    except Exception as error:

        xai_status = (
            f"Grad-CAM unavailable: "
            f"{type(error).__name__}: {error}"
        )

    return (
        summary,
        dataframe,
        chart,
        overlay,
        heatmap,
        xai_status,
        result
    )


def create_report(result, image):

    if not result:
        raise gr.Error(
            "Run image analysis before generating a report."
        )

    image_name = "uploaded_eye_image.png"

    report = generate_report(
        result=result,
        image_name=image_name,
        save=True
    )

    return report["text"], report["path"]


CUSTOM_CSS = """
.gradio-container {
    max-width: 1350px !important;
    margin: auto !important;
}
#hero {
    text-align: center;
    padding: 18px 0;
}
#hero h1 {
    font-size: 2.5rem;
    font-weight: 800;
    color: #B9A6FF;
}
#hero p {
    color: #BFC7DC;
}
"""

with gr.Blocks(
    title="NetraScan.AI",
    css=CUSTOM_CSS,
    theme=gr.themes.Soft(
        primary_hue="violet",
        secondary_hue="indigo",
        neutral_hue="slate"
    )
) as demo:

    gr.HTML(
        """
        <div id="hero">
            <h1>NetraScan.AI</h1>
            <p>
                Multi-Transformer Cataract Classification
                &amp; Explainable AI
            </p>
        </div>
        """
    )

    prediction_state = gr.State(value=None)

    with gr.Row():

        with gr.Column(scale=1):

            image_input = gr.Image(
                label="Eye Image",
                sources=["upload", "webcam"],
                type="pil",
                height=340
            )

            analyze_button = gr.Button(
                "Analyze Eye Image",
                variant="primary",
                size="lg"
            )

            clear_button = gr.Button(
                "Clear Results",
                variant="secondary"
            )

        with gr.Column(scale=1):

            ensemble_output = gr.Markdown(
                "## Awaiting Image Analysis"
            )

            prediction_table = gr.Dataframe(
                label="Individual Model Predictions",
                interactive=False
            )

    probability_chart = gr.Plot(
        label="Class Probability Distribution"
    )

    gr.Markdown("## Explainable AI — Swin-Tiny Grad-CAM")

    with gr.Row():

        overlay_output = gr.Image(
            label="Grad-CAM Overlay",
            interactive=False,
            height=310
        )

        heatmap_output = gr.Image(
            label="Activation Heatmap",
            interactive=False,
            height=310
        )

    xai_message = gr.Markdown()

    gr.Markdown("## Research Report")

    report_button = gr.Button(
        "Generate Classification Report",
        variant="primary"
    )

    report_preview = gr.Markdown()

    report_download = gr.File(
        label="Download Markdown Report",
        interactive=False
    )

    gr.Markdown(
        """
        ---
        **Research disclaimer:** NetraScan.AI is an
        academic prototype. Predictions and Grad-CAM
        visualizations are not medical diagnoses.
        """
    )

    analyze_button.click(
        fn=run_analysis,
        inputs=[image_input],
        outputs=[
            ensemble_output,
            prediction_table,
            probability_chart,
            overlay_output,
            heatmap_output,
            xai_message,
            prediction_state
        ]
    )

    report_button.click(
        fn=create_report,
        inputs=[
            prediction_state,
            image_input
        ],
        outputs=[
            report_preview,
            report_download
        ]
    )

    clear_button.click(
        fn=lambda: (
            None, "## Awaiting Image Analysis",
            None, None, None, None, "",
            None, "", None
        ),
        inputs=[],
        outputs=[
            image_input,
            ensemble_output,
            prediction_table,
            probability_chart,
            overlay_output,
            heatmap_output,
            xai_message,
            prediction_state,
            report_preview,
            report_download
        ]
    )


if __name__ == "__main__":

    print("=" * 65)
    print("NETRASCAN.AI — PORTABLE APPLICATION")
    print("Project:", APP_ROOT)
    print("=" * 65)

    initialize()

    demo.launch(
        server_name="127.0.0.1",
        server_port=7860,
        inbrowser=True,
        share=False,
        allowed_paths=[str(REPORTS_DIR)]
    )
'''

print("\n1. VALIDATING APPLICATION SOURCE")
print("-" * 65)

ast.parse(app_source)
print("PASS — Python syntax validated.")

print("\n2. WRITING STANDALONE APPLICATION")
print("-" * 65)

APP_FILE.write_text(
    app_source.lstrip(),
    encoding="utf-8"
)

print("Created:", APP_FILE)
print("Size:", round(APP_FILE.stat().st_size / 1024, 2), "KB")

print("\n3. VERIFYING APPLICATION FEATURES")
print("-" * 65)

features = {
    "Three-model inference": "NetraScanInference",
    "Swin-Tiny Grad-CAM": "NetraScanXAI",
    "Offline report generator": "generate_report",
    "Image upload and webcam": 'sources=["upload", "webcam"]',
    "Ensemble prediction": "ensemble_prediction",
    "Probability comparison chart": "create_probability_chart",
    "Downloadable Markdown report": "report_download",
    "Local-only server": 'server_name="127.0.0.1"'
}

for feature, keyword in features.items():
    assert keyword in app_source
    print(f"PASS — {feature}")

print("\nPASS — Standalone Gradio application created.")
print("Next: Run and test the complete application.")


---

## Project description and conclusion

**NetraScan.AI** is a multi-transformer, explainable AI research prototype for three-class cataract image classification. The study began with a ViT-B/16 baseline and uncovered potentially misleading validation performance caused by related images crossing data partitions. To address this, the project investigated perceptual similarity and augmentation lineage, then constructed a V4 visual-family-aware split for a more defensible comparison.

Three models were fine-tuned and evaluated on the same held-out V4 test partition. **Swin-Tiny achieved the highest reported test accuracy (98.95%)**, followed by **MobileViT-S (98.25%)** and **ViT-B/16 (97.37%)**. MobileViT-S offers a substantially smaller architecture, while Swin-Tiny was selected for qualitative Grad-CAM visualization using an intermediate feature stage. A separate standalone Gradio prototype combines all three model predictions using majority voting and provides visual explanations and downloadable structured Markdown reports.

### Key contributions

- Investigation and mitigation of duplicate-image and augmentation-lineage leakage.
- Controlled comparison of global-attention, shifted-window, and hybrid mobile transformer architectures.
- Per-model evaluation, error-family analysis, and cross-model comparison.
- Swin-Tiny Grad-CAM and a portable demonstration application.
- A transparent, **template-based offline reporting engine**. The current deterministic report generator is **not** a generative language model; LLM-based report generation is a future extension.

### Limitations and future work

The evaluation is based on a limited, internally curated image collection. Family-aware partitioning reduces identified duplicate leakage but does not guarantee patient-level independence or external clinical validity. Model scores require calibration and external validation; Grad-CAM activations do not establish clinically correct anatomical localization. Future work includes independent multi-site evaluation, robustness testing across imaging conditions, confidence calibration, and careful integration of a genuine LLM-assisted reporting component with appropriate safeguards.

### Repository notes

This notebook is a curated experimental record. The source image datasets and large model checkpoints are not embedded in the notebook. The standalone Python application should be committed as separate `.py` files alongside an environment specification and a README. Update machine-specific `ROOT` and SSD destination paths before executing cells. Do not commit patient-identifying images, proprietary datasets, or unreviewed large model weights.

---

**Created by Mr. SHOBHIT JAIN**  
**Project:** NetraScan.AI — Multi-Transformer Cataract Classification and Explainable AI  
**Purpose:** Academic research, reproducibility, and prototype demonstration
